[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wdittaya/AI-for-digital-health-course/blob/main/lab08_llm_prompt_engineering.ipynb)

# ปฏิบัติการที่ 8: แบบจำลองภาษาขนาดใหญ่และการออกแบบพรอมต์ (Prompt Engineering) สำหรับข้อความทางคลินิก

**รายวิชา:** 3099704 AI for Digital Health (2026) · **คาบเรียน:** วันเสาร์ที่ 17 ตุลาคม 2026 (ภาคบ่าย) · **เวลาโดยประมาณ:** 90 นาที

## วัตถุประสงค์การเรียนรู้

- สามารถรันแบบจำลองภาษาขนาดใหญ่ (large language model, LLM) แบบเปิดบน Colab โดยไม่ต้องใช้ API key และไม่มีค่าใช้จ่าย
- สามารถออกแบบพรอมต์ (prompt) สำหรับงานทางคลินิก ได้แก่ การสรุปความ (summarization) การสกัดข้อมูล (information extraction) และการถาม-ตอบ (question answering)
- สามารถกำหนดให้แบบจำลองตอบเป็นข้อมูลมีโครงสร้าง (JSON) เพื่อนำไปใช้งานต่อ
- ตระหนักถึงการสร้างข้อมูลเท็จ (hallucination) และข้อจำกัดด้านความปลอดภัยของ LLM ในงานสุขภาพ

> **แนวทางการทำปฏิบัติการ** (งานเดี่ยว)
>
> ✅ **ส่วนหลัก — สำหรับผู้เรียนทุกคน ไม่จำเป็นต้องมีพื้นฐานการเขียนโปรแกรม** ให้รันเซลล์ (cell) ตามลำดับจากบนลงล่าง (กด Shift+Enter หรือคลิกปุ่ม ▶) ในปฏิบัติการนี้ ผู้เรียนเพียงแก้ไขข้อความพรอมต์ภายในเครื่องหมายอัญประกาศ `"..."` แล้วรันเซลล์ใหม่ ส่วนโค้ดอื่นได้จัดเตรียมไว้แล้ว แบบฝึกหัดทุกข้อมีเซลล์ ✅ **เฉลย** อยู่ถัดไป ให้ผู้เรียนทำแบบฝึกหัดด้วยตนเองก่อน หากไม่สามารถดำเนินการต่อได้ ให้รันเซลล์เฉลยแล้วทำขั้นตอนถัดไป จากนั้นให้ตอบคำถาม ✍️ ในเซลล์ข้อความ ซึ่งเป็นชิ้นงานที่ต้องส่ง หากพบปัญหา สามารถสอบถามผู้สอนผ่านช่องแชตของ Zoom
>
> 🏆 **โจทย์ท้าทายบนตารางอันดับ — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้** อยู่ท้ายโน้ตบุ๊ก (notebook) และไม่นับเป็นส่วนหนึ่งของชิ้นงานที่ต้องส่ง

> ⚡ **จำเป็นต้องใช้ GPU:** ก่อนเริ่มปฏิบัติการ ให้เลือก `Runtime → Change runtime type → T4 GPU → Save` (ไม่มีค่าใช้จ่าย) หากไม่ใช้ GPU แบบจำลองยังทำงานได้ แต่แต่ละคำตอบจะใช้เวลาประมาณ 1 นาที

> 🔒 **จริยธรรม:** ห้ามนำข้อมูลผู้ป่วยจริงไปป้อนให้แบบจำลองปัญญาประดิษฐ์ (AI) หรือเว็บไซต์ใด ๆ บันทึกทางคลินิกทั้งหมดในปฏิบัติการนี้เป็นข้อมูลสังเคราะห์

> 🌐 **ภาษาที่ใช้:** บันทึกทางคลินิกและพรอมต์ที่ส่งให้แบบจำลองในปฏิบัติการนี้เป็นภาษาอังกฤษ เนื่องจากแบบจำลองขนาดเล็กทำงานได้ดีที่สุดกับภาษาอังกฤษ และชุดข้อมูล (dataset) เป็นภาษาอังกฤษ ทุกเซลล์มีคำอธิบายภาษาไทยกำกับความหมายของพรอมต์ และมีเซลล์สำหรับทดลองสั่งงานเป็นภาษาไทยเพื่อเปรียบเทียบ

In [ ]:
# @title ⚙️ การตั้งค่า — โปรดรันเซลล์นี้ก่อน (NAME และ JOIN_CODE ใช้เฉพาะโจทย์ท้าทายบนตารางอันดับ ซึ่งไม่บังคับ)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้เป็นการ "เตรียมความพร้อม" ก่อนเริ่มปฏิบัติการ โดยทำ 3 อย่าง ได้แก่
#   1) ติดตั้งเครื่องมือ (ไลบรารี) เพิ่มเติมที่จำเป็นต้องใช้ใน Google Colab
#   2) กำหนดค่าพื้นฐาน เช่น รหัสปฏิบัติการ ชื่อที่จะแสดง และรหัสเข้าร่วมตารางอันดับ
#   3) สร้าง "ฟังก์ชัน" (ชุดคำสั่งสำเร็จรูปที่เรียกใช้ซ้ำได้) 2 ตัว คือ
#        - lab_data(...)              : ดาวน์โหลดไฟล์ข้อมูลของรายวิชา
#        - submit_to_leaderboard(...) : ส่งผลการทำนายไปยังตารางอันดับ (ไม่บังคับ)
#   ผู้เรียนเพียงกดรันเซลล์นี้ 1 ครั้ง ไม่จำเป็นต้องแก้ไขโค้ด
#   (ยกเว้นต้องการเข้าร่วมตารางอันดับ ให้กรอก NAME และ JOIN_CODE ด้านล่าง)
# ============================================================================

# ติดตั้งไลบรารีเพิ่มเติมลงในเครื่อง Colab
#   - เครื่องหมาย ! ด้านหน้า หมายถึงสั่งงานระบบปฏิบัติการ (ไม่ใช่คำสั่งภาษา Python)
#   - pip install : คำสั่งติดตั้งไลบรารีของ Python
#   - -q (quiet)  : ให้แสดงข้อความระหว่างติดตั้งให้น้อยที่สุด
#   - gradio_client   : ไลบรารีสำหรับติดต่อกับเว็บแอปตารางอันดับ (Hugging Face Space)
#   - huggingface_hub : ไลบรารีสำหรับดาวน์โหลดไฟล์จากเว็บไซต์ Hugging Face
!pip -q install gradio_client huggingface_hub

# สร้างตัวแปร (ช่องเก็บค่า) HF_OWNER แล้วใส่ข้อความ "wdittaya" ไว้
#   เครื่องหมาย = หมายถึง "นำค่าทางขวา ไปเก็บไว้ในชื่อทางซ้าย"
#   ข้อความในภาษา Python ต้องอยู่ในเครื่องหมายคำพูด " "
HF_OWNER  = "wdittaya"     # บัญชี Hugging Face ที่ใช้โฮสต์ตารางอันดับของรายวิชา (ผู้สอนเป็นผู้กำหนด)
# รหัสของปฏิบัติการนี้ ใช้ระบุว่าผลที่ส่งเป็นของปฏิบัติการใด (ไม่ต้องแก้ไข)
LAB_ID    = "lab08"
# ชื่อที่จะแสดงบนตารางอันดับ: พิมพ์ชื่อไว้ระหว่างเครื่องหมาย " " เช่น NAME = "หมอเอ"
NAME      = ""            # ไม่บังคับ: ชื่อที่จะแสดงบนตารางอันดับแบบสาธารณะ (สามารถใช้นามแฝงได้)
# รหัสเข้าร่วมตารางอันดับ: พิมพ์รหัสที่ผู้สอนแจ้งไว้ระหว่างเครื่องหมาย " "
JOIN_CODE = ""            # ไม่บังคับ: รหัสเข้าร่วมซึ่งผู้สอนประกาศในการเรียนการสอนผ่าน Zoom

# สร้างชื่อเว็บแอปตารางอันดับ โดยนำ HF_OWNER มาต่อกับข้อความ
#   - f"..." (f-string) คือข้อความที่แทรกค่าตัวแปรได้ โดยเขียนชื่อตัวแปรไว้ใน { }
#   - ผลลัพธ์คือ "wdittaya/aidh-leaderboard"
SPACE_ID  = f"{HF_OWNER}/aidh-leaderboard"
# สร้างชื่อคลังข้อมูล (dataset) ของรายวิชาในลักษณะเดียวกัน ผลลัพธ์คือ "wdittaya/aidh-lab-data"
DATA_REPO = f"{HF_OWNER}/aidh-lab-data"

# นำเข้า (import) ฟังก์ชัน hf_hub_download จากไลบรารี huggingface_hub
#   ฟังก์ชันนี้ใช้ดาวน์โหลดไฟล์จากเว็บไซต์ Hugging Face มาเก็บไว้ในเครื่อง
from huggingface_hub import hf_hub_download

# ----------------------------------------------------------------------------
# สร้างฟังก์ชัน lab_data สำหรับดาวน์โหลดไฟล์ข้อมูลของรายวิชา
#   - def คือคำสั่งสร้างฟังก์ชันใหม่ ตามด้วยชื่อฟังก์ชันและพารามิเตอร์ในวงเล็บ
#   - path : พารามิเตอร์ (ค่าที่ผู้ใช้ส่งเข้ามา) คือชื่อไฟล์ที่ต้องการ เช่น "pima/test_ids.csv"
#   - บรรทัดที่ย่อหน้าเข้าไปด้านล่าง คือคำสั่งที่อยู่ภายในฟังก์ชัน
# ----------------------------------------------------------------------------
def lab_data(path):
    """ดาวน์โหลดไฟล์ข้อมูลทดสอบสาธารณะของรายวิชา 1 ไฟล์ (เช่น 'pima/test_ids.csv') และคืนค่าตำแหน่งไฟล์ในเครื่อง"""
    # เรียก hf_hub_download(...) เพื่อดาวน์โหลดไฟล์ แล้วส่งตำแหน่งไฟล์ในเครื่องกลับไป (return)
    #   - DATA_REPO           : ชื่อคลังข้อมูลบน Hugging Face ที่เก็บไฟล์
    #   - path                : ชื่อไฟล์ที่ต้องการดาวน์โหลดภายในคลังข้อมูลนั้น
    #   - repo_type="dataset" : ระบุว่าคลังนี้เป็นประเภท "ชุดข้อมูล" (ไม่ใช่โมเดล)
    return hf_hub_download(DATA_REPO, path, repo_type="dataset")

# ----------------------------------------------------------------------------
# สร้างฟังก์ชัน submit_to_leaderboard สำหรับส่งไฟล์ผลการทำนายไปยังตารางอันดับ
#   - path   : ชื่อไฟล์ผลการทำนายที่ต้องการส่ง
#   - lab=LAB_ID : รหัสปฏิบัติการ หากไม่ระบุ จะใช้ค่า LAB_ID ที่กำหนดไว้ด้านบนโดยอัตโนมัติ
# ----------------------------------------------------------------------------
def submit_to_leaderboard(path, lab=LAB_ID):
    """ส่งไฟล์ผลการทำนายไปยังตารางอันดับของรายวิชา และแสดงคะแนนพร้อมอันดับ"""
    # นำเข้าเครื่องมือ 2 ตัวจากไลบรารี gradio_client
    #   - Client      : ใช้เชื่อมต่อกับเว็บแอปตารางอันดับ
    #   - handle_file : ใช้เตรียมไฟล์ในเครื่องให้อยู่ในรูปแบบที่ส่งขึ้นเว็บแอปได้
    from gradio_client import Client, handle_file
    # assert คือการตรวจสอบเงื่อนไข หากเงื่อนไขเป็นเท็จ โปรแกรมจะหยุดและแสดงข้อความเตือน
    #   - NAME and JOIN_CODE : เงื่อนไขว่าต้องกรอกทั้ง NAME และ JOIN_CODE แล้ว (ไม่เป็นข้อความว่าง)
    #   - ข้อความหลังเครื่องหมายจุลภาค คือข้อความเตือนที่จะแสดงเมื่อยังไม่ได้กรอก
    assert NAME and JOIN_CODE, "โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่าก่อน (จำเป็นเฉพาะการส่งผลไปยังตารางอันดับ)"
    # เชื่อมต่อกับเว็บแอปตารางอันดับ แล้วเก็บการเชื่อมต่อไว้ในตัวแปร client
    #   - SPACE_ID      : ชื่อเว็บแอปตารางอันดับที่กำหนดไว้ด้านบน
    #   - verbose=False : ไม่ต้องแสดงข้อความรายละเอียดระหว่างเชื่อมต่อ
    client = Client(SPACE_ID, verbose=False)
    # ส่งข้อมูลไปยังเว็บแอปด้วย client.predict(...) แล้วแสดงผลลัพธ์ (คะแนนและอันดับ) ด้วย print(...)
    #   - lab=lab                 : รหัสปฏิบัติการ
    #   - name=NAME               : ชื่อที่จะแสดงบนตารางอันดับ
    #   - code=JOIN_CODE          : รหัสเข้าร่วม
    #   - file=handle_file(path)  : ไฟล์ผลการทำนายที่เตรียมพร้อมสำหรับส่ง
    #   - api_name="/submit"      : ชื่อช่องทางรับข้อมูลของเว็บแอป (ส่วน "ส่งผล")
    print(client.predict(lab=lab, name=NAME, code=JOIN_CODE, file=handle_file(path), api_name="/submit"))

# แสดงข้อความยืนยันว่าการตั้งค่าเสร็จแล้ว ด้วยฟังก์ชัน print(...) (พิมพ์ข้อความออกทางหน้าจอ)
#   - ข้อความเป็น f-string จึงแทรกค่า SPACE_ID, LAB_ID และ NAME ลงในข้อความได้
#   - NAME or '(ยังไม่ได้ระบุ ...)' : หาก NAME ว่าง จะแสดงข้อความในวงเล็บแทน
print(f"Leaderboard: https://huggingface.co/spaces/{SPACE_ID}  |  lab={LAB_ID}  name={NAME or '(ยังไม่ได้ระบุ — ตารางอันดับไม่บังคับ)'}")

## 1. การโหลดแบบจำลองภาษาขนาดใหญ่แบบเปิดบน Colab

**แบบจำลองภาษาขนาดใหญ่ (LLM)** คือโปรแกรมที่ผ่านการฝึกจากข้อความจำนวนมากเพื่อทำนายคำถัดไป ส่วน LLM ชนิด *instruction-tuned* ได้รับการฝึกเพิ่มเติมให้ปฏิบัติตามคำสั่งในลักษณะผู้ช่วยสนทนา ปฏิบัติการนี้ใช้แบบจำลอง **แบบเปิด (open model)** ขนาดเล็กจาก Hugging Face ชื่อ `Qwen2.5-1.5B-Instruct` (1.5 พันล้านพารามิเตอร์) ซึ่งทำงานบน GPU T4 ของ Colab รุ่นไม่มีค่าใช้จ่ายได้ โดยไม่ต้องใช้ API key และข้อความไม่ถูกส่งออกนอกเซสชัน Colab ของผู้เรียน หลักการออกแบบพรอมต์ในปฏิบัติการนี้สามารถนำไปใช้กับแบบจำลองขนาดใหญ่กว่าได้เช่นกัน (ChatGPT, Claude, Gemini เป็นต้น)

ให้รันเซลล์ถัดไป การรันครั้งแรกจะดาวน์โหลดแบบจำลอง (ประมาณ 3 GB) และใช้เวลาประมาณ **2–4 นาที**

In [ ]:
# @title 🤖 โหลดแบบจำลองและฟังก์ชันช่วย `chat()` — ให้รันเซลล์นี้โดยไม่ต้องแก้ไข
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้ "เตรียมแบบจำลองภาษาขนาดใหญ่ (large language model, LLM)" ให้พร้อมใช้งาน โดยทำ 4 อย่าง ได้แก่
#   1) ติดตั้งไลบรารี transformers และ accelerate ซึ่งใช้ดาวน์โหลดและรันแบบจำลอง
#   2) ตรวจสอบว่าเครื่อง Colab มีหน่วยประมวลผลกราฟิก (GPU) หรือไม่ (GPU ช่วยให้ตอบเร็วขึ้นมาก)
#   3) ดาวน์โหลดแบบจำลองแบบเปิด Qwen2.5-1.5B-Instruct จาก Hugging Face มาไว้ในเครื่อง Colab
#      (ข้อความทางคลินิกจึงไม่ถูกส่งออกไปยังบริการภายนอก)
#   4) สร้างฟังก์ชันช่วย chat(system, user) สำหรับ "ถาม" แบบจำลอง 1 ครั้ง แล้วรับคำตอบเป็นข้อความ
#   ผลลัพธ์ที่ผู้เรียนจะเห็น: ข้อความแจ้งว่าพบ GPU หรือไม่ และคำตอบตัวอย่าง 1 ประโยคเกี่ยวกับ hypertension
#   (การดาวน์โหลดแบบจำลองครั้งแรกอาจใช้เวลา 1–3 นาที)
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: ติดตั้งและนำเข้าไลบรารี
# ----------------------------------------------------------------------------
# ติดตั้งไลบรารีเพิ่มเติมลงในเครื่อง Colab
#   - เครื่องหมาย ! ด้านหน้า หมายถึงสั่งงานระบบปฏิบัติการ (ไม่ใช่คำสั่งภาษา Python)
#   - pip install : คำสั่งติดตั้งไลบรารีของ Python
#   - -q (quiet)  : แสดงข้อความระหว่างติดตั้งให้น้อยที่สุด
#   - "transformers>=4.56" : ไลบรารีของ Hugging Face สำหรับใช้งานแบบจำลองภาษา
#                            โดย >=4.56 หมายถึงต้องเป็นรุ่น 4.56 ขึ้นไป (ใส่ในเครื่องหมายคำพูดเพราะมีสัญลักษณ์ >)
#   - accelerate : ไลบรารีช่วยจัดวางแบบจำลองลงบน GPU/CPU โดยอัตโนมัติ (ใช้ร่วมกับ device_map="auto" ด้านล่าง)
!pip -q install "transformers>=4.56" accelerate
# นำเข้า (import) ไลบรารี 2 ตัวในบรรทัดเดียว (คั่นด้วยเครื่องหมายจุลภาค)
#   - torch        : ไลบรารี PyTorch สำหรับการคำนวณของแบบจำลองเชิงลึก และตรวจสอบ GPU
#   - transformers : ไลบรารีของ Hugging Face ที่เพิ่งติดตั้งด้านบน
import torch, transformers
# นำเข้าฟังก์ชัน pipeline จากไลบรารี transformers
#   pipeline คือ "ชุดเครื่องมือสำเร็จรูป" ที่รวมการโหลดแบบจำลอง การแปลงข้อความ และการสร้างคำตอบไว้ในคำสั่งเดียว
from transformers import pipeline
# ตั้งค่าให้ไลบรารี transformers แสดงเฉพาะข้อความระดับ "ข้อผิดพลาด (error)" เท่านั้น
#   - transformers.logging.set_verbosity_error() : ปิดคำเตือนทางเทคนิคที่ไม่จำเป็น เพื่อให้ผลลัพธ์อ่านง่าย
#   - ข้อความหลังเครื่องหมาย # ในบรรทัดเดียวกัน คือหมายเหตุ (comment) ที่ Python ไม่นำไปทำงาน
transformers.logging.set_verbosity_error()   # ซ่อนคำเตือนทางเทคนิค

# ----------------------------------------------------------------------------
# ส่วนที่ 2: ตรวจสอบว่ามี GPU หรือไม่
# ----------------------------------------------------------------------------
# if ... else ... คือคำสั่งเลือกทำงานตามเงื่อนไข: หากเงื่อนไขเป็นจริงจะทำคำสั่งที่ย่อหน้าใต้ if
#   หากเป็นเท็จจะทำคำสั่งที่ย่อหน้าใต้ else
#   - torch.cuda.is_available() : ฟังก์ชันของ PyTorch ที่คืนค่า True หากเครื่องนี้มี GPU (CUDA) ให้ใช้ และ False หากไม่มี
if torch.cuda.is_available():
    # แสดงข้อความว่าพบ GPU พร้อมชื่อรุ่นของ GPU ด้วย print(...)
    #   - print สามารถรับหลายค่าโดยคั่นด้วยจุลภาค และจะแสดงต่อกันโดยเว้นวรรค 1 ช่อง
    #   - torch.cuda.get_device_name(0) : คืนชื่อรุ่นของ GPU ตัวแรก (หมายเลข 0) เช่น "Tesla T4"
    print("✅ พบ GPU:", torch.cuda.get_device_name(0))
# กรณีไม่พบ GPU
else:
    # แสดงคำเตือนวิธีเปิดใช้ GPU ใน Colab
    #   - ข้อความ 2 ชุดที่วางติดกันในวงเล็บ Python จะนำมาต่อกันเป็นข้อความเดียวโดยอัตโนมัติ
    #   - \n คือสัญลักษณ์ "ขึ้นบรรทัดใหม่" ภายในข้อความ
    print("⚠️ ไม่พบ GPU — โปรดเลือก Runtime → Change runtime type → T4 GPU → Save แล้วรันทุกเซลล์ใหม่\n"
          "   (สามารถทำต่อบน CPU ได้ แต่แต่ละคำตอบจะใช้เวลาประมาณ 1 นาที)")

# ----------------------------------------------------------------------------
# ส่วนที่ 3: ดาวน์โหลดและโหลดแบบจำลอง
# ----------------------------------------------------------------------------
# เก็บชื่อแบบจำลองบน Hugging Face ไว้ในตัวแปร MODEL
#   - "Qwen/Qwen2.5-1.5B-Instruct" : แบบจำลอง Qwen2.5 ขนาด 1.5 พันล้านพารามิเตอร์ ชนิด Instruct
#     (ผ่านการฝึกเพิ่มเติมให้ปฏิบัติตามคำสั่งแบบผู้ช่วยสนทนา)
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
# สร้าง "ตัวสร้างข้อความ" ด้วย pipeline(...) แล้วเก็บไว้ในตัวแปร llm (คำสั่งนี้เขียนต่อกัน 3 บรรทัด)
#   - "text-generation" : ชนิดของงาน คือการสร้างข้อความต่อจากข้อความที่ป้อนเข้าไป
#   - model=MODEL       : ชื่อแบบจำลองที่จะดาวน์โหลดและใช้งาน
#   - dtype=...         : ชนิดตัวเลขที่ใช้เก็บค่าของแบบจำลอง เขียนแบบมีเงื่อนไขในบรรทัดเดียว
#                         (รูปแบบ "ค่า A if เงื่อนไข else ค่า B")
#       * torch.float16 : ตัวเลขทศนิยม 16 บิต ใช้หน่วยความจำครึ่งหนึ่งและเร็วกว่า เลือกเมื่อมี GPU
#       * torch.float32 : ตัวเลขทศนิยม 32 บิต ใช้เมื่อรันบน CPU (CPU ทำงานกับ float16 ได้ไม่ดี)
#   - device_map="auto" : ให้ไลบรารี accelerate เลือกวางแบบจำลองบน GPU หากมี มิฉะนั้นใช้ CPU
llm = pipeline("text-generation", model=MODEL,
               dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
               device_map="auto")

# ----------------------------------------------------------------------------
# ส่วนที่ 4: สร้างฟังก์ชันช่วย chat(...) สำหรับถามแบบจำลอง 1 ครั้ง
#   - def คือคำสั่งสร้างฟังก์ชันใหม่ ตามด้วยชื่อฟังก์ชันและพารามิเตอร์ (ค่าที่รับเข้ามา) ในวงเล็บ
#   - system : ข้อความกำหนดบทบาทและกฎของแบบจำลอง (system prompt)
#   - user   : คำขอหรือคำถามของผู้ใช้ (user prompt)
#   - max_new_tokens=300 : จำนวนโทเคน (token คือคำหรือส่วนของคำ) สูงสุดที่แบบจำลองจะตอบ
#                          เครื่องหมาย = ในวงเล็บหมายถึง "ค่าเริ่มต้น" หากไม่ระบุจะใช้ 300
#   - examples=None : (ไม่บังคับ) รายการตัวอย่างคู่ (คำขอ, คำตอบ) สำหรับเทคนิค few-shot
#                     None หมายถึง "ไม่มีค่า" คือไม่ใส่ตัวอย่าง
# ----------------------------------------------------------------------------
def chat(system, user, max_new_tokens=300, examples=None):
    """ส่งคำขอ 1 ครั้งไปยังแบบจำลอง และคืนค่าคำตอบเป็นข้อความ
    system = บทบาทและกฎของแบบจำลอง;  user = คำขอ;  examples = (ไม่บังคับ) รายการตัวอย่างคู่ (คำขอ, คำตอบ)"""
    # สร้างรายการ (list) ข้อความสนทนาชื่อ messages โดยเริ่มจากข้อความ system 1 รายการ
    #   - [ ... ] คือ list (รายการที่เก็บได้หลายค่าตามลำดับ)
    #   - { ... } คือ dictionary (ชุดข้อมูลแบบ "คีย์: ค่า")
    #   - "role": "system" : ระบุว่าข้อความนี้เป็นของ "ระบบ" (กำหนดบทบาท/กฎ)
    #   - "content": system : เนื้อหาข้อความ คือค่าของพารามิเตอร์ system
    messages = [{"role": "system", "content": system}]
    # วนซ้ำ (for loop) ผ่านตัวอย่าง few-shot แต่ละคู่ เพื่อใส่ไว้ในบทสนทนาก่อนคำถามจริง
    #   - (examples or []) : หาก examples เป็น None จะใช้ list ว่าง [] แทน (คือไม่วนเลย)
    #   - ex_user, ex_reply : แยกแต่ละคู่ออกเป็น "คำขอตัวอย่าง" และ "คำตอบตัวอย่าง"
    for ex_user, ex_reply in (examples or []):
        # เพิ่มข้อความ 2 รายการต่อท้าย messages (+= หมายถึง "นำมาต่อท้ายของเดิม")
        #   - role "user"      : คำขอตัวอย่าง
        #   - role "assistant" : คำตอบตัวอย่างที่ต้องการ (ให้แบบจำลองเห็นรูปแบบที่ถูกต้อง)
        messages += [{"role": "user", "content": ex_user}, {"role": "assistant", "content": ex_reply}]
    # เพิ่มคำขอจริงของผู้ใช้ต่อท้ายบทสนทนา ด้วยเมธอด .append(...) ของ list (เพิ่ม 1 รายการต่อท้าย)
    messages.append({"role": "user", "content": user})
    # ส่งบทสนทนาให้แบบจำลองสร้างคำตอบ แล้วเก็บผลไว้ในตัวแปร out
    #   - messages : บทสนทนาทั้งหมด (system + ตัวอย่าง + คำขอ)
    #   - max_new_tokens=max_new_tokens : จำกัดความยาวคำตอบตามค่าที่รับเข้ามา
    #   - do_sample=False : ไม่สุ่ม ให้เลือกคำที่น่าจะเป็นที่สุดเสมอ พรอมต์เดิมจึงได้คำตอบเดิมทุกครั้ง
    #                       (ทำให้เปรียบเทียบพรอมต์ต่าง ๆ ได้อย่างเป็นธรรม)
    out = llm(messages, max_new_tokens=max_new_tokens, do_sample=False)
    # ดึงเฉพาะข้อความคำตอบออกมา แล้วส่งกลับ (return) ไปยังผู้เรียกฟังก์ชัน
    #   - out[0] : ผลลัพธ์ชุดแรก (pipeline คืนผลเป็น list)
    #   - ["generated_text"] : บทสนทนาทั้งหมดรวมคำตอบใหม่ของแบบจำลองที่ต่อท้าย
    #   - [-1] : ข้อความรายการสุดท้าย (ดัชนี -1 หมายถึงตัวสุดท้าย) คือคำตอบของ assistant
    #   - ["content"] : เนื้อหาข้อความของคำตอบนั้น
    return out[0]["generated_text"][-1]["content"]

# ทดสอบฟังก์ชัน chat(...) และแสดงคำตอบด้วย print(...)
#   - "You are a helpful medical assistant." : system = "คุณคือผู้ช่วยทางการแพทย์ที่เป็นประโยชน์"
#   - "In one sentence, what is hypertension?" : user = "จงอธิบายว่า hypertension คืออะไร ใน 1 ประโยค"
print(chat("You are a helpful medical assistant.", "In one sentence, what is hypertension?"))

💡 **คำอธิบายผลลัพธ์**
เซลล์ข้างต้นดาวน์โหลดแบบจำลองและจัดเตรียมฟังก์ชันช่วย `chat(system, user)` ข้อความ **system** กำหนดบทบาทและกฎของแบบจำลอง ส่วนข้อความ **user** คือคำขอ (ตัวอย่างข้างต้นขอให้อธิบาย hypertension ใน 1 ประโยค) พารามิเตอร์ `max_new_tokens` จำกัดความยาวของคำตอบ (*token* คือคำหรือส่วนของคำ) และ `do_sample=False` ทำให้พรอมต์เดิมให้คำตอบเดิมทุกครั้ง จึงสามารถเปรียบเทียบพรอมต์ต่าง ๆ ได้อย่างเป็นธรรม

In [ ]:
# @title 🧰 ฟังก์ชันช่วยสำหรับจัดการคำตอบแบบ JSON — ให้รันเซลล์นี้โดยไม่ต้องแก้ไข
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้สร้าง "ฟังก์ชันช่วย" สำหรับจัดการคำตอบของแบบจำลองที่อยู่ในรูปแบบ JSON
#   (JSON คือรูปแบบข้อความมาตรฐานสำหรับข้อมูลมีโครงสร้าง (structured data) เช่น {"age": 57, "sex": "M"})
#   แบบจำลองภาษามักตอบ JSON ปนกับข้อความอื่น เช่น ```json หรือคำอธิบายเพิ่มเติม จึงต้องมีตัวช่วยคัดแยก ได้แก่
#     - parse_json_reply(raw) : ค้นหาบล็อก {...} ในคำตอบ แล้วแปลงเป็น dictionary ของ Python
#     - _as_list(v)           : (ตัวช่วยภายใน) แปลงค่ารูปแบบต่าง ๆ ให้เป็นรายการข้อความ (list)
#     - clean_record(d)       : จัดข้อมูลให้เหลือเฉพาะ age / sex / conditions / medications ในรูปแบบมาตรฐาน
#     - show_record(data)     : แสดงข้อมูลที่สกัดได้ให้อ่านง่าย และแจ้งคีย์ที่ขาดหายไป
#   ผู้เรียนเพียงรันเซลล์นี้ 1 ครั้ง จะเห็นข้อความยืนยันว่าฟังก์ชันพร้อมใช้งาน
# ============================================================================

# นำเข้าไลบรารีมาตรฐานของ Python 2 ตัว
#   - json : ใช้แปลงข้อความ JSON <-> ข้อมูลของ Python (dictionary, list)
#   - re   : (regular expression) ใช้ค้นหา/แทนที่ข้อความตาม "รูปแบบ" ที่กำหนด
import json, re

# ----------------------------------------------------------------------------
# ฟังก์ชัน parse_json_reply : ดึง JSON ออกจากคำตอบของแบบจำลอง
#   - raw         : ข้อความคำตอบดิบจากแบบจำลอง
#   - quiet=False : หากเป็น False จะแสดงคำเตือนเมื่อหา JSON ไม่พบ (ค่าเริ่มต้น); True = ไม่แสดงคำเตือน
# ----------------------------------------------------------------------------
def parse_json_reply(raw, quiet=False):
    """ค้นหาบล็อก {...} แรกในคำตอบของแบบจำลอง (ข้ามเครื่องหมาย ```json และข้อความส่วนเกิน)
    และแปลงเป็น dictionary ของ Python คืนค่า {} หากไม่พบ JSON ที่ถูกต้อง"""
    # ลบเครื่องหมาย ``` และ ```json ออกจากคำตอบ แล้วเก็บผลไว้ในตัวแปร text
    #   - re.sub(รูปแบบ, ข้อความใหม่, ข้อความเดิม) : แทนที่ทุกส่วนที่ตรงรูปแบบด้วยข้อความใหม่
    #   - r"```(?:json)?" : รูปแบบ "``` ที่อาจตามด้วยคำว่า json หรือไม่ก็ได้" (? = มีหรือไม่มีก็ได้)
    #                       ตัว r หน้าข้อความหมายถึง raw string (ไม่ตีความเครื่องหมาย \ เป็นพิเศษ)
    #   - ""       : แทนที่ด้วยข้อความว่าง คือลบทิ้ง
    #   - str(raw) : แปลง raw ให้เป็นข้อความก่อน (กันกรณีไม่ใช่ข้อความ)
    text = re.sub(r"```(?:json)?", "", str(raw))
    # สร้างตัวถอดรหัส JSON (JSONDecoder) จากไลบรารี json เก็บไว้ในตัวแปร decoder
    #   ตัวนี้มีเมธอด raw_decode ที่อ่าน JSON จากต้นข้อความได้ แม้มีข้อความอื่นตามหลัง
    decoder = json.JSONDecoder()
    # วนซ้ำทุกตำแหน่งที่พบเครื่องหมาย { ในข้อความ (จุดเริ่มต้นที่อาจเป็น JSON)
    #   - re.finditer(r"\{", text) : ค้นหาทุกตำแหน่งที่ตรงรูปแบบ และให้ผลทีละรายการ
    #   - r"\{" : เครื่องหมาย { (ต้องใส่ \ นำหน้า เพราะ { มีความหมายพิเศษในรูปแบบ regular expression)
    #   - m : ผลการค้นหาแต่ละรายการ (match object)
    for m in re.finditer(r"\{", text):
        # try ... except ... : ลองทำคำสั่งใน try หากเกิดข้อผิดพลาดตามชนิดที่ระบุ จะไปทำคำสั่งใน except แทน
        #   (ป้องกันไม่ให้โปรแกรมหยุดทำงานเมื่อข้อความบางส่วนไม่ใช่ JSON ที่ถูกต้อง)
        try:
            # พยายามแปลงข้อความตั้งแต่ตำแหน่ง { นี้เป็นต้นไปให้เป็นข้อมูล Python
            #   - m.start() : ตำแหน่ง (ลำดับตัวอักษร) ที่พบเครื่องหมาย {
            #   - text[m.start():] : ตัดข้อความตั้งแต่ตำแหน่งนั้นไปจนจบ (slicing แบบ [เริ่ม:])
            #   - decoder.raw_decode(...) : คืนค่า 2 อย่าง คือ (ข้อมูลที่แปลงได้, ตำแหน่งที่อ่านจบ)
            #   - obj, _ : เก็บข้อมูลไว้ใน obj ส่วนตำแหน่งที่อ่านจบไม่ใช้ จึงเก็บในชื่อ _ (ชื่อที่นิยมใช้แทน "ไม่สนใจ")
            obj, _ = decoder.raw_decode(text[m.start():])
            # ตรวจว่าสิ่งที่แปลงได้เป็น dictionary จริงหรือไม่
            #   - isinstance(obj, dict) : คืน True หาก obj เป็นชนิด dict
            if isinstance(obj, dict):
                # หากใช่ ส่ง dictionary นี้กลับทันที (return จะจบการทำงานของฟังก์ชัน)
                return obj
        # หากแปลงไม่สำเร็จ (เกิด json.JSONDecodeError คือข้อความไม่ใช่ JSON ที่ถูกต้อง)
        except json.JSONDecodeError:
            # continue : ข้ามไปลองตำแหน่ง { ถัดไปในรอบถัดไปของ loop
            continue
    # หากวนครบแล้วยังไม่พบ JSON: ตรวจว่าต้องแสดงคำเตือนหรือไม่ (not quiet = quiet เป็น False)
    if not quiet:
        # แสดงคำเตือนแนะนำให้ปรับพรอมต์
        print("⚠️ ไม่พบ JSON ที่ถูกต้องในคำตอบ — โปรดปรับพรอมต์ให้กำหนดรูปแบบอย่างเข้มงวดขึ้น")
    # คืนค่า dictionary ว่าง {} เพื่อให้โค้ดส่วนอื่นทำงานต่อได้โดยไม่เกิดข้อผิดพลาด
    return {}

# ----------------------------------------------------------------------------
# ฟังก์ชัน _as_list : (ตัวช่วยภายใน — ชื่อขึ้นต้นด้วย _ ตามธรรมเนียมว่าไม่ได้ออกแบบให้ผู้ใช้เรียกโดยตรง)
#   แปลงค่า v ที่แบบจำลองตอบมาในรูปแบบต่าง ๆ ให้กลายเป็น list ของข้อความเสมอ
#   - v : ค่าที่ต้องการแปลง เช่น None, ข้อความ "a, b", dictionary หรือ list
# ----------------------------------------------------------------------------
def _as_list(v):
    # กรณีที่ 1: ไม่มีค่า (None) — ตรวจด้วย is None
    if v is None:
        # คืน list ว่าง
        return []
    # กรณีที่ 2: v เป็นข้อความ (str) เช่น "metformin; amlodipine"
    if isinstance(v, str):
        # แยกข้อความด้วยเครื่องหมาย ; หรือ , แล้วตัดช่องว่างหัวท้าย และทิ้งรายการว่าง
        #   - re.split(r"[;,]", v) : แบ่งข้อความ v ทุกตำแหน่งที่พบ ; หรือ , ([...] = ตัวใดตัวหนึ่งในวงเล็บ)
        #   - s.strip() : ตัดช่องว่างหัวท้ายของแต่ละส่วน
        #   - [ ... for s in ... if s.strip()] : list comprehension คือการสร้าง list ใหม่ด้วยการวนในบรรทัดเดียว
        #     โดยเก็บเฉพาะส่วนที่ไม่ว่าง (if s.strip() เป็นจริงเมื่อมีตัวอักษรเหลืออยู่)
        return [s.strip() for s in re.split(r"[;,]", v) if s.strip()]
    # กรณีที่ 3: v เป็น dictionary — ใช้เฉพาะ "ค่า" ของแต่ละคีย์
    if isinstance(v, dict):
        # v.values() : ดึงค่าทั้งหมดใน dictionary แล้วแปลงเป็น list ด้วย list(...)
        v = list(v.values())
    # กรณีที่ 4: v ไม่ใช่ list (เช่น ตัวเลข) — not isinstance(...) คือ "ไม่ใช่ชนิด list"
    if not isinstance(v, list):
        # แปลงเป็นข้อความด้วย str(v) แล้วห่อเป็น list 1 รายการ
        return [str(v)]
    # กรณีที่ 5: v เป็น list แล้ว — แปลงทุกรายการให้เป็นข้อความที่ตัดช่องว่างหัวท้าย (list comprehension)
    #   - for x in v if x : วนทุกรายการ x ใน v โดยข้ามรายการที่ว่าง/None
    #   - x.get("name", x.get("drug", "")) if isinstance(x, dict) else x :
    #       หาก x เป็น dictionary (เช่น {"name": "metformin", "dose": "500mg"}) ให้ใช้ค่าของคีย์ "name"
    #       หากไม่มีคีย์ "name" ให้ใช้คีย์ "drug" และหากไม่มีทั้งคู่ให้ใช้ข้อความว่าง ""
    #       (.get(คีย์, ค่าสำรอง) คือดึงค่าจาก dictionary โดยคืนค่าสำรองหากไม่พบคีย์)
    #       หาก x ไม่ใช่ dictionary ให้ใช้ x ตามเดิม
    #   - str(...).strip() : แปลงเป็นข้อความและตัดช่องว่างหัวท้าย
    return [str(x.get("name", x.get("drug", "")) if isinstance(x, dict) else x).strip() for x in v if x]

# ----------------------------------------------------------------------------
# ฟังก์ชัน clean_record : จัดข้อมูลให้อยู่ในรูปแบบมาตรฐานที่ตารางอันดับกำหนด
#   - d : dictionary ที่ได้จาก parse_json_reply(...)
#   ผลลัพธ์: dictionary ที่มีเฉพาะ 4 คีย์ คือ age (จำนวนเต็มหรือ None), sex ("M"/"F"/None),
#            conditions (list ของโรค) และ medications (list ของชื่อยา)
# ----------------------------------------------------------------------------
def clean_record(d):
    """คงไว้เฉพาะ age / sex / conditions / medications ในรูปแบบที่ตารางอันดับกำหนด"""
    # หาก d เป็น dictionary ให้ใช้ตามเดิม มิฉะนั้นให้ใช้ dictionary ว่าง {} แทน (กันข้อผิดพลาด)
    d = d if isinstance(d, dict) else {}
    # กำหนดค่าเริ่มต้นของอายุเป็น None (ยังไม่ทราบ)
    age = None
    # ค้นหาตัวเลขชุดแรกในค่าอายุ (เช่น "57 years" -> "57")
    #   - d.get("age", "") : ดึงค่าคีย์ "age" หากไม่มีให้ใช้ข้อความว่าง
    #   - str(...) : แปลงเป็นข้อความก่อนค้นหา
    #   - re.search(r"\d+", ...) : ค้นหาตำแหน่งแรกที่ตรงรูปแบบ \d+ (ตัวเลข 0-9 ตั้งแต่ 1 หลักขึ้นไป)
    #     คืนผลการค้นหา (match) หากพบ หรือ None หากไม่พบ
    m = re.search(r"\d+", str(d.get("age", "")))
    # หากพบตัวเลข (m ไม่เป็น None จะถือเป็นจริง)
    if m:
        # m.group(0) : ข้อความส่วนที่ตรงรูปแบบทั้งหมด เช่น "57" แล้วแปลงเป็นจำนวนเต็มด้วย int(...)
        age = int(m.group(0))
    # ดึงค่าเพศ แปลงเป็นข้อความ ตัดช่องว่างหัวท้าย (.strip()) และแปลงเป็นตัวพิมพ์เล็ก (.lower())
    #   เช่น " Male " -> "male" เพื่อให้เทียบกับตารางด้านล่างได้ง่าย
    sex = str(d.get("sex", "")).strip().lower()
    # แปลงคำเรียกเพศหลากหลายแบบให้เป็นรหัสมาตรฐาน "M" หรือ "F"
    #   - { ... } : dictionary ที่จับคู่คำ -> รหัส เช่น "male" -> "M", "woman" -> "F"
    #   - .get(sex) : ค้นหาค่าของ sex ในตาราง หากไม่พบจะได้ None (ไม่ทราบเพศ)
    sex = {"m": "M", "male": "M", "man": "M", "f": "F", "female": "F", "woman": "F"}.get(sex)
    # สร้างและคืน dictionary ผลลัพธ์ 4 คีย์ (คำสั่งเขียนต่อกัน 2 บรรทัดภายในวงเล็บปีกกา)
    #   - "age": age, "sex": sex : ค่าที่จัดรูปแบบแล้วด้านบน
    #   - "conditions" : แปลงค่าโรคให้เป็น list ด้วย _as_list(...)
    #   - "medications" : แปลงค่ายาให้เป็น list ด้วย _as_list(...)
    return {"age": age, "sex": sex,
            "conditions": _as_list(d.get("conditions")), "medications": _as_list(d.get("medications"))}

# ----------------------------------------------------------------------------
# ฟังก์ชัน show_record : แสดงข้อมูลที่สกัดได้ และตรวจว่ามีคีย์ที่จำเป็นครบหรือไม่
#   - data : dictionary ที่ต้องการแสดง
#   - required=(...) : ชื่อคีย์ที่จำเป็น (ค่าเริ่มต้นคือ age, sex, conditions, medications)
#                      (...) แบบนี้เรียกว่า tuple คือรายการที่แก้ไขไม่ได้
# ----------------------------------------------------------------------------
def show_record(data, required=("age", "sex", "conditions", "medications")):
    """แสดงข้อมูลที่สกัดได้ในรูปแบบที่อ่านง่าย และระบุคีย์ที่ขาดหายไป"""
    # ตรวจว่า data ยังเป็นช่องว่าง ... (ผู้เรียนยังไม่ได้เติมในแบบฝึกหัด) หรือไม่ใช่ dictionary
    #   - data is ... : ตรวจว่า data คือค่า ... (Ellipsis) ซึ่งใช้เป็น "ช่องว่างให้เติม" ในแบบฝึกหัด
    #   - or : เงื่อนไขใดเงื่อนไขหนึ่งเป็นจริงก็พอ
    if data is ... or not isinstance(data, dict):
        # แสดงคำเตือน (เครื่องหมาย ` ` ในข้อความเป็นเพียงตัวอักษร ไม่มีผลกับโค้ด)
        print("⚠️ `data` ยังไม่เป็น dictionary — โปรดตรวจสอบว่าได้แทนที่ ... แล้ว")
        # return เปล่า ๆ : จบฟังก์ชันทันทีโดยไม่คืนค่า
        return
    # แสดง data เป็นข้อความ JSON ที่จัดย่อหน้าสวยงาม
    #   - json.dumps(...) : แปลงข้อมูล Python เป็นข้อความ JSON
    #   - indent=2 : ย่อหน้าแต่ละระดับ 2 ช่องว่าง เพื่อให้อ่านง่าย
    #   - ensure_ascii=False : แสดงอักษรที่ไม่ใช่ภาษาอังกฤษ (เช่น ภาษาไทย) ตามจริง แทนรหัส \uXXXX
    print(json.dumps(data, indent=2, ensure_ascii=False))
    # หาคีย์ที่จำเป็นแต่ไม่มีใน data (list comprehension)
    #   - for k in required : วนชื่อคีย์ที่จำเป็นทีละตัว
    #   - if k not in data : เก็บเฉพาะคีย์ที่ไม่อยู่ใน data
    missing = [k for k in required if k not in data]
    # แสดงผลการตรวจ โดยเลือกข้อความแบบมีเงื่อนไข ("ข้อความ A" if เงื่อนไข else "ข้อความ B")
    #   - not missing : เป็นจริงเมื่อ list missing ว่าง (ไม่มีคีย์ขาด) -> แสดง "มีคีย์ที่จำเป็นครบถ้วน"
    #   - มิฉะนั้นแสดงรายชื่อคีย์ที่ขาด โดยใช้ f-string แทรกค่า {missing}
    #   - \n ด้านหน้า : เว้นบรรทัดก่อนแสดงข้อความ
    print("\n✅ มีคีย์ที่จำเป็นครบถ้วน" if not missing else f"\n⚠️ คีย์ที่ขาดหายไป: {missing}")

# แสดงข้อความยืนยันว่าฟังก์ชันช่วยทั้งหมดพร้อมใช้งานแล้ว
print("ฟังก์ชันช่วยพร้อมใช้งาน: parse_json_reply(), clean_record(), show_record()")

## 2. องค์ประกอบของพรอมต์ (prompt patterns)

พรอมต์ทางคลินิกที่ดีควรระบุองค์ประกอบ 4 ประการ ดังนี้

| องค์ประกอบ | ตัวอย่าง (ภาษาอังกฤษที่ใช้ในปฏิบัติการ) |
|---|---|
| **บทบาท (role)** — แบบจำลองทำหน้าที่ใด | "You are a clinical documentation assistant." (ผู้ช่วยจัดทำเวชระเบียน) |
| **งาน (task)** — ให้ดำเนินการสิ่งใด | "Summarize this note." (สรุปบันทึกนี้) |
| **ข้อจำกัด (constraints)** — กฎและขอบเขต | "Use only facts in the note. Do not guess." (ใช้เฉพาะข้อมูลในบันทึก ห้ามคาดเดา) |
| **รูปแบบผลลัพธ์ (output format)** — ลักษณะของคำตอบ | "Exactly 3 bullet points." / "Return JSON." (3 หัวข้อพอดี / ตอบเป็น JSON) |

เซลล์ถัดไปใช้พรอมต์ **ที่ไม่ชัดเจน** (ระบุเพียงบทบาทและงาน) กับบันทึกการติดตามอาการสังเคราะห์ 1 ฉบับ (ผู้ป่วยชายอายุ 57 ปี โรคเบาหวานชนิดที่ 2)

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้เตรียม "บันทึกทางคลินิก (clinical note)" สังเคราะห์ 1 ฉบับ เก็บไว้ในตัวแปร note
#   (ใช้ต่อเนื่องในแบบฝึกหัดทุกข้อของปฏิบัติการนี้) แล้วทดลองสั่งแบบจำลองด้วย "พรอมต์ที่ไม่ชัดเจน"
#   คือสั่งเพียง "Summarize" (สรุป) โดยไม่ระบุบทบาท ผู้อ่าน รูปแบบ หรือข้อจำกัด
#   ผลลัพธ์ที่ผู้เรียนจะเห็น: สรุปจากพรอมต์ที่ไม่ชัดเจน ซึ่งจะใช้เปรียบเทียบกับพรอมต์ที่ปรับปรุงในแบบฝึกหัดที่ 1
#   ความหมายของบันทึก: ชายอายุ 57 ปี เป็นเบาหวานชนิดที่ 2 (T2DM) มาติดตามอาการ ปัสสาวะบ่อยและอ่อนเพลีย 2 สัปดาห์
#   ความดันโลหิต 148/92 น้ำตาลขณะอดอาหาร 142 mg/dL HbA1c 8.1% ใช้ metformin 500mg วันละ 2 ครั้ง (BID)
#   ไม่มีเจ็บหน้าอก แผน: เพิ่มขนาด metformin ให้คำแนะนำการปรับพฤติกรรม และตรวจซ้ำใน 6 สัปดาห์
# ============================================================================

# เก็บข้อความบันทึกทางคลินิกไว้ในตัวแปร note
#   - """ ... """ (อัญประกาศ 3 ตัว) คือข้อความหลายบรรทัด (multi-line string) การขึ้นบรรทัดใหม่จะถูกเก็บไว้ด้วย
note = """57M with T2DM presents for follow-up. Reports polyuria and fatigue x2 weeks.
BP 148/92. Fasting glucose 142 mg/dL, HbA1c 8.1%. On metformin 500mg BID.
No chest pain. Plan: increase metformin, add lifestyle counseling, recheck in 6 weeks."""

# แสดงหัวข้อคั่น เพื่อให้รู้ว่าผลด้านล่างมาจากพรอมต์ที่ไม่ชัดเจน
print("--- พรอมต์ที่ไม่ชัดเจน ---")
# ถามแบบจำลองด้วยฟังก์ชัน chat(...) (สร้างไว้ในเซลล์โหลดแบบจำลอง) แล้วแสดงคำตอบด้วย print(...)
#   - "You are a clinical assistant." : system = "คุณคือผู้ช่วยทางคลินิก" (บทบาทแบบกว้าง ๆ)
#   - f"Summarize: {note}" : user = "สรุป: " ตามด้วยบันทึก
#       f-string (ข้อความที่ขึ้นต้นด้วย f) แทรกค่าตัวแปรที่อยู่ใน { } ลงในข้อความ ที่นี่คือแทรกเนื้อหาของ note
print(chat("You are a clinical assistant.", f"Summarize: {note}"))

In [ ]:
# ⭐ Exercise 1 — แบบฝึกหัดที่ 1: การเขียนพรอมต์สรุปบันทึกให้มีคุณภาพดีขึ้น
# เป้าหมายทางคลินิก: สรุปบันทึกอย่างกระชับและปลอดภัย สำหรับส่งต่อแพทย์ปฐมภูมิ (primary-care physician)
# ✏️ สิ่งที่ต้องทำ: แก้ไขเฉพาะข้อความภายในเครื่องหมายอัญประกาศ (เขียนเป็นภาษาอังกฤษ) แล้วรันเซลล์ (Shift+Enter)
#   1. ใน `system` ให้กำหนดบทบาทของแบบจำลอง
#   2. ใน `user` ให้ระบุจำนวนหัวข้อของสรุป ผู้อ่าน และข้อกำหนดให้ใช้เฉพาะข้อมูลที่ปรากฏในบันทึก
# 💡 คำใบ้: ตัวอย่างเช่น "Write exactly 3 bullet points for a primary-care physician. Use only information
#    in the note; do not add anything."  ห้ามลบ {note} เนื่องจากเป็นตำแหน่งที่บันทึกจะถูกแทรก
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   ฝึกเขียนพรอมต์ที่มีองค์ประกอบครบ ได้แก่ บทบาท (role) งาน (task) ข้อจำกัด (constraints)
#   และรูปแบบผลลัพธ์ (output format) แล้วส่งให้แบบจำลองสรุปบันทึก note จากเซลล์ก่อนหน้า
#   ผลลัพธ์ที่ผู้เรียนจะเห็น: สรุปจากพรอมต์ของผู้เรียน เพื่อนำไปเปรียบเทียบกับสรุปจากพรอมต์ที่ไม่ชัดเจน
#   หากยังไม่ได้แก้ไขช่อง ✏️ เซลล์จะแสดงคำเตือนก่อน
# ============================================================================

# ข้อความ system (system prompt) : กำหนดบทบาทของแบบจำลอง
#   - "You are ✏️ ..." : ส่วน ✏️ ... คือช่องว่างที่ผู้เรียนต้องแทนที่ด้วยบทบาท (ภาษาอังกฤษ) ตามข้อ 1 ด้านบน
system = "You are ✏️ ..."

# ข้อความ user (user prompt) : คำสั่งงานพร้อมบันทึกที่ต้องสรุป
#   - f"""...""" : f-string หลายบรรทัด ซึ่งแทรกค่าตัวแปรใน { } ได้
#   - บรรทัด "✏️ Write your instructions here." คือช่องที่ผู้เรียนต้องเขียนคำสั่ง (ภาษาอังกฤษ) ตามข้อ 2 ด้านบน
#   - "Note:" ตามด้วย {note} : ตำแหน่งที่เนื้อหาบันทึกจะถูกแทรก (ห้ามลบ {note})
user = f"""✏️ Write your instructions here.

Note:
{note}"""

# ตรวจว่ายังมีเครื่องหมาย ✏️ หลงเหลืออยู่หรือไม่ (แปลว่ายังไม่ได้แก้ไขช่องว่าง)
#   - system + user : นำข้อความ 2 ชุดมาต่อกัน (+ กับข้อความคือการต่อข้อความ)
#   - "✏️" in ... : เป็นจริงหากพบเครื่องหมาย ✏️ ในข้อความ
if "✏️" in system + user:
    # แสดงคำเตือนให้แก้ไขช่อง ✏️ ก่อน (\n ท้ายข้อความ = เว้นบรรทัด)
    print("⚠️ โปรดแทนที่ส่วนที่มีเครื่องหมาย ✏️ ด้วยข้อความของผู้เรียน (หรือรันเซลล์เฉลยด้านล่าง)\n")
# ส่ง system และ user ให้แบบจำลองด้วย chat(...) แล้วแสดงสรุปที่ได้ด้วย print(...)
print(chat(system, user))

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ความหมายของพรอมต์: บทบาท = ผู้ช่วยจัดทำเวชระเบียนที่ไม่เพิ่มเติมข้อมูลเอง; สรุปสำหรับแพทย์ปฐมภูมิ 3 หัวข้อ
# (ผู้ป่วยและปัญหาหลัก / สิ่งตรวจพบที่สำคัญ / แผนการรักษา) ใช้เฉพาะข้อมูลในบันทึก ความยาวไม่เกิน 60 คำ
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เฉลยแบบฝึกหัดที่ 1: ตัวอย่างพรอมต์ที่มีองค์ประกอบครบทั้ง 4 ประการ
#     - บทบาท (role)           : ผู้ช่วยจัดทำเวชระเบียนที่มีประสบการณ์
#     - งาน (task)              : สรุปบันทึกสำหรับแพทย์ปฐมภูมิ
#     - ข้อจำกัด (constraints)  : ใช้เฉพาะข้อมูลในบันทึก ห้ามสร้างค่า การวินิจฉัย หรือยาขึ้นเอง
#     - รูปแบบผลลัพธ์ (format) : 3 หัวข้อพอดี ความยาวไม่เกิน 60 คำ
#   ผลลัพธ์ที่ผู้เรียนจะเห็น: สรุป 3 หัวข้อที่กระชับกว่าและยึดตามบันทึกมากกว่าพรอมต์ที่ไม่ชัดเจน
# ============================================================================

# ข้อความ system : กำหนดบทบาทและนิสัยของแบบจำลอง
#   ความหมาย: "คุณคือผู้ช่วยจัดทำเวชระเบียนที่มีประสบการณ์ เขียนสรุปที่กระชับและถูกต้อง
#             และไม่เพิ่มข้อมูลที่ไม่มีในบันทึกต้นฉบับ"
#   - ข้อความ 2 ชุดในวงเล็บที่วางติดกัน Python จะนำมาต่อกันเป็นข้อความเดียว (ช่วยให้เขียนข้อความยาวหลายบรรทัดได้)
system = ("You are an experienced clinical documentation assistant. You write concise, accurate summaries "
          "and never add information that is not in the source note.")
# ข้อความ user : คำสั่งงานพร้อมกฎ (f-string หลายบรรทัด แทรกบันทึกที่ตำแหน่ง {note})
#   ความหมาย: "สรุปบันทึกด้านล่างสำหรับแพทย์ปฐมภูมิ โดยมีกฎ:
#     - 3 หัวข้อพอดี: (1) ผู้ป่วยและปัญหาหลัก (2) สิ่งตรวจพบที่สำคัญ (3) แผนการรักษา
#     - ใช้เฉพาะข้อมูลที่ระบุในบันทึก ห้ามสร้างค่า การวินิจฉัย หรือยาขึ้นเอง
#     - ความยาวไม่เกิน 60 คำ"
user = f"""Summarize the clinical note below for a primary-care physician.
Rules:
- Exactly 3 bullet points: (1) patient and main problem, (2) key findings, (3) plan.
- Use ONLY information stated in the note. Do not invent values, diagnoses or medications.
- Keep it under 60 words.

Note:
{note}"""
# ส่งพรอมต์ให้แบบจำลองด้วย chat(system, user) แล้วแสดงสรุปที่ได้ด้วย print(...)
print(chat(system, user))

💡 **คำอธิบายผลลัพธ์**
การแก้ไขเพียงถ้อยคำของพรอมต์ โดยไม่เปลี่ยนโค้ด ทำให้ผลลัพธ์เปลี่ยนไป การระบุ **ผู้อ่าน** **รูปแบบ** และข้อจำกัด **"ใช้เฉพาะข้อมูลในบันทึก"** มักทำให้ได้สรุปที่กระชับ สม่ำเสมอ และปลอดภัยกว่า กระบวนการนี้เรียกว่า *การออกแบบพรอมต์ (prompt engineering)*

🌐 **การทดลองเพิ่มเติม (ไม่บังคับ):** เซลล์ถัดไปสั่งงานเดียวกันเป็น **ภาษาไทย** ให้รันเซลล์และเปรียบเทียบกับผลลัพธ์ภาษาอังกฤษ แบบจำลองขนาดเล็กมักตอบภาษาไทยได้ไม่คล่องหรือคลาดเคลื่อนมากกว่า ผู้เรียนสามารถแก้ไขข้อความภาษาไทยภายในเครื่องหมายอัญประกาศได้

In [ ]:
# 🌐 การทดลองเพิ่มเติม (ไม่บังคับ): สั่งงานเป็นภาษาไทย แล้วเปรียบเทียบกับผลลัพธ์ภาษาอังกฤษข้างต้น
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   ทดลองสั่งงานเดียวกับแบบฝึกหัดที่ 1 (สรุปบันทึก 3 หัวข้อ ใช้เฉพาะข้อมูลในบันทึก) แต่เขียนพรอมต์เป็นภาษาไทย
#   และขอให้ตอบเป็นภาษาไทย เพื่อดูว่าแบบจำลองขนาดเล็กทำงานกับภาษาไทยได้ดีเพียงใด
#   ผลลัพธ์ที่ผู้เรียนจะเห็น: สรุปภาษาไทย ซึ่งอาจไม่คล่องหรือคลาดเคลื่อนมากกว่าผลลัพธ์ภาษาอังกฤษ
#   ผู้เรียนสามารถแก้ไขข้อความภาษาไทยภายในเครื่องหมายอัญประกาศได้
# ============================================================================

# สร้างคำสั่งภาษาไทยเก็บไว้ในตัวแปร thai_user
#   - f"""...""" : f-string หลายบรรทัด โดย {note} คือตำแหน่งที่เนื้อหาบันทึก (ภาษาอังกฤษ) จะถูกแทรก
thai_user = f"""กรุณาสรุปบันทึกทางคลินิกด้านล่างเป็นภาษาไทยจำนวน 3 หัวข้อ สำหรับแพทย์เวชปฏิบัติทั่วไป
โดยใช้เฉพาะข้อมูลที่ปรากฏในบันทึก และไม่เพิ่มเติมข้อมูลอื่น

บันทึก:
{note}"""
# ถามแบบจำลองด้วย chat(...) แล้วแสดงคำตอบ
#   - "You are a helpful clinical assistant. Answer in Thai." : system = "คุณคือผู้ช่วยทางคลินิก ให้ตอบเป็นภาษาไทย"
#   - thai_user : คำสั่งภาษาไทยที่สร้างไว้ด้านบน
print(chat("You are a helpful clinical assistant. Answer in Thai.", thai_user))

✍️ **คำถามที่ 1:** จงเปรียบเทียบสรุปจากพรอมต์ที่ไม่ชัดเจนกับสรุปจากพรอมต์ที่ปรับปรุงแล้ว โปรดระบุความแตกต่าง 2 ประการ และระบุว่ามีสรุปใดที่เพิ่มข้อมูลซึ่ง **ไม่ปรากฏ** ในบันทึกหรือไม่ (หากได้ทดลองพรอมต์ภาษาไทย โปรดอธิบายความแตกต่างของผลลัพธ์โดยสังเขป)

✍️ **คำตอบ:**



## 3. ผลลัพธ์แบบมีโครงสร้าง (JSON)

ข้อความอิสระเหมาะสำหรับผู้อ่านที่เป็นมนุษย์ แต่ระบบเวชระเบียนอิเล็กทรอนิกส์ (electronic health record, EHR) ทะเบียนผู้ป่วย (registry) และแดชบอร์ด (dashboard) ต้องการ **ข้อมูลมีโครงสร้าง (structured data)** **JSON** เป็นรูปแบบข้อความมาตรฐานสำหรับจุดประสงค์นี้ ประกอบด้วยช่องข้อมูลที่มีชื่อกำกับภายในวงเล็บปีกกา เช่น `{"age": 57, "sex": "M", "medications": ["metformin"]}` ในส่วนนี้จะกำหนดให้แบบจำลองตอบเป็น JSON *เท่านั้น* แล้วใช้ฟังก์ชันช่วย `parse_json_reply()` แปลงคำตอบเป็น dictionary ของ Python ฟังก์ชันนี้รองรับข้อความส่วนเกิน เช่น เครื่องหมาย ```` ```json ```` ซึ่งแบบจำลองขนาดเล็กมักแทรกมาด้วย

In [ ]:
# ⭐ Exercise 2 — แบบฝึกหัดที่ 2: การสกัดข้อมูลเป็นโครงสร้าง (JSON)
# เป้าหมายทางคลินิก: แปลงบันทึกข้อความอิสระเป็นระเบียนข้อมูลมีโครงสร้างที่ระบบคอมพิวเตอร์นำไปใช้ได้
# ความหมายของพรอมต์ (ภาษาอังกฤษ): "ให้ตอบเป็น JSON เท่านั้น โดยมีคีย์ age, sex, conditions, vitals, plan"
# ✏️ สิ่งที่ต้องทำ (แก้ไข 2 ตำแหน่ง):
#   1. พรอมต์ขาดคีย์ 1 รายการ ให้เพิ่ม  medications (list)  ในรายการคีย์ (ภายในเครื่องหมายอัญประกาศ)
#   2. ที่บรรทัด `data = ...` ให้แทนที่ ... ด้วย  parse_json_reply(raw)
# 💡 คำใบ้: parse_json_reply เป็นฟังก์ชันช่วยที่จัดเตรียมไว้แล้ว ทำหน้าที่ค้นหาบล็อก {...} ในคำตอบ
#    และแปลงเป็น dictionary
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   ฝึกสั่งให้แบบจำลอง "สกัดข้อมูล (information extraction)" จากบันทึก note ออกมาเป็น JSON
#   ซึ่งเป็นข้อมูลมีโครงสร้าง (structured data) ที่ระบบเวชระเบียนอิเล็กทรอนิกส์ (EHR) หรือทะเบียนผู้ป่วยนำไปใช้ต่อได้
#   ขั้นตอน: (1) ส่งพรอมต์ให้แบบจำลอง  (2) แสดงคำตอบดิบ  (3) แปลงคำตอบเป็น dictionary  (4) แสดงผลและตรวจคีย์ที่ขาด
#   ผลลัพธ์ที่ผู้เรียนจะเห็น: คำตอบดิบของแบบจำลอง ตามด้วยข้อมูลที่จัดรูปแบบแล้ว และผลการตรวจคีย์
# ============================================================================

# ข้อความ system : กำหนดบทบาทและรูปแบบการตอบ
#   ความหมาย: "คุณคือผู้สกัดข้อมูลทางคลินิก ให้ตอบเป็น JSON ที่ถูกต้องเท่านั้น ไม่มีข้อความบรรยาย"
system = "You are a clinical information extractor. Respond with ONLY valid JSON, no prose."
# ข้อความ user : คำสั่งสกัดข้อมูล พร้อมรายชื่อคีย์และชนิดข้อมูลที่ต้องการ (f-string หลายบรรทัด แทรกบันทึกที่ {note})
#   ความหมาย: "สกัดข้อมูลมีโครงสร้างจากบันทึกนี้เป็น JSON โดยมีคีย์ดังนี้:
#     age (จำนวนเต็ม), sex ("M" หรือ "F"), conditions (รายการ), vitals (ออบเจกต์), plan (รายการ)"
#   - ✏️ ผู้เรียนต้องเพิ่มคีย์ที่ขาดไป 1 รายการในบรรทัดรายชื่อคีย์ ตามข้อ 1 ด้านบน
user = f"""Extract structured data from this note as JSON with these keys:
age (integer), sex ("M" or "F"), conditions (list), vitals (object), plan (list)

Note:
{note}"""

# ส่งพรอมต์ให้แบบจำลอง แล้วเก็บคำตอบดิบ (ข้อความ) ไว้ในตัวแปร raw
#   - system, user : พรอมต์ทั้งสองส่วนที่สร้างไว้ด้านบน
#   - max_new_tokens=300 : ให้ตอบได้ยาวสูงสุด 300 โทเคน (เพียงพอสำหรับ JSON ขนาดนี้)
raw = chat(system, user, max_new_tokens=300)
# แสดงคำตอบดิบของแบบจำลอง
#   - print รับ 3 ค่า: ข้อความหัวเรื่อง (มี \n ขึ้นบรรทัดใหม่), raw และ "\n" (เว้นบรรทัดท้าย) โดยคั่นด้วยช่องว่าง
print("คำตอบของแบบจำลอง:\n", raw, "\n")

# แปลงคำตอบดิบให้เป็น dictionary ของ Python แล้วเก็บไว้ในตัวแปร data
#   - ... (จุด 3 จุด) คือ "ช่องว่าง" ที่ผู้เรียนต้องเติม ตามข้อ 2 และคำใบ้ด้านบน
#   - หมายเหตุท้ายบรรทัดเดิมระบุสิ่งที่ต้องเติมไว้แล้ว
data = ...   # ✏️ แทนที่ ... ด้วย parse_json_reply(raw)

# แสดงข้อมูลใน data ให้อ่านง่าย และตรวจว่ามีคีย์ age / sex / conditions / medications ครบหรือไม่
#   - show_record(...) : ฟังก์ชันช่วยจากเซลล์ 🧰 ด้านบน (หาก data ยังเป็น ... จะแสดงคำเตือน)
show_record(data)

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เฉลยแบบฝึกหัดที่ 2: พรอมต์สกัดข้อมูลที่เพิ่มคีย์ medications (list) แล้ว
#   และใช้ parse_json_reply(raw) แปลงคำตอบดิบของแบบจำลองให้เป็น dictionary
#   ผลลัพธ์ที่ผู้เรียนจะเห็น: คำตอบดิบ, ข้อมูล JSON ที่จัดรูปแบบแล้ว, ผลการตรวจคีย์ และรายการยาที่สกัดได้
#   (ผู้เรียนควรตรวจทุกช่องข้อมูลเทียบกับบันทึก เพราะแบบจำลองอาจผิดพลาดได้)
# ============================================================================

# ข้อความ system : "คุณคือผู้สกัดข้อมูลทางคลินิก ให้ตอบเป็น JSON ที่ถูกต้องเท่านั้น ไม่มีข้อความบรรยาย"
system = "You are a clinical information extractor. Respond with ONLY valid JSON, no prose."
# ข้อความ user : คำสั่งสกัดข้อมูล (f-string หลายบรรทัด แทรกบันทึกที่ {note})
#   ความหมาย: "สกัดข้อมูลมีโครงสร้างจากบันทึกนี้เป็น JSON โดยมีคีย์: age (จำนวนเต็ม), sex ("M" หรือ "F"),
#             conditions (รายการโรค), medications (รายการยา), vitals (ออบเจกต์สัญญาณชีพ), plan (รายการแผนการรักษา)"
#   - จุดที่เฉลย: เพิ่ม medications (list) ในรายการคีย์
user = f"""Extract structured data from this note as JSON with these keys:
age (integer), sex ("M" or "F"), conditions (list), medications (list), vitals (object), plan (list)

Note:
{note}"""

# ส่งพรอมต์ให้แบบจำลองด้วย chat(...) แล้วเก็บคำตอบดิบไว้ใน raw
#   - max_new_tokens=300 : ความยาวคำตอบสูงสุด 300 โทเคน
raw = chat(system, user, max_new_tokens=300)
# แสดงคำตอบดิบ (หัวเรื่อง + raw + บรรทัดว่าง)
print("คำตอบของแบบจำลอง:\n", raw, "\n")

# จุดที่เฉลย: แปลงคำตอบดิบเป็น dictionary ด้วยฟังก์ชันช่วย parse_json_reply(raw) แล้วเก็บใน data
#   - raw : ข้อความคำตอบของแบบจำลอง (ฟังก์ชันจะข้าม ```json และข้อความส่วนเกิน แล้วหาบล็อก {...})
data = parse_json_reply(raw)

# แสดงข้อมูลให้อ่านง่าย และตรวจว่ามีคีย์ที่จำเป็นครบหรือไม่
show_record(data)
# แสดงเฉพาะรายการยาที่สกัดได้
#   - data.get("medications") : ดึงค่าของคีย์ "medications" จาก dictionary
#     (หากไม่มีคีย์นี้จะได้ None แทนการเกิดข้อผิดพลาด ต่างจากการเขียน data["medications"])
print("\nรายการยา (medications):", data.get("medications"))

💡 **คำอธิบายผลลัพธ์**
แบบจำลองแปลงบันทึกที่เขียนเป็นประโยคให้เป็นช่องข้อมูลที่มีชื่อกำกับ เมื่อ `data` เป็น dictionary แล้ว โปรแกรมสามารถนำไปใช้ต่อได้ทันที เช่น `data["medications"]` ให้รายการยา ซึ่งอาจส่งต่อไปยังระบบตรวจสอบอันตรกิริยาระหว่างยา (drug interaction) หรือทะเบียนผู้ป่วยเบาหวาน อย่างไรก็ตาม แบบจำลองยังอาจผิดพลาด จึงต้องตรวจสอบทุกช่องข้อมูล

✍️ **คำถามที่ 2:** จงตรวจสอบระเบียนที่สกัดได้เทียบกับบันทึกทีละบรรทัด โรค (conditions) และยา (medications) ถูกต้องและครบถ้วนหรือไม่ มีข้อมูลใดตกหล่น หรือถูก **สร้างขึ้นเอง** หรือไม่ (เช่น การวินิจฉัย hypertension ซึ่งบันทึกไม่ได้ระบุไว้)

✍️ **คำตอบ:**



## 4. การสร้างข้อมูลเท็จ (hallucination) และความปลอดภัย

**การสร้างข้อมูลเท็จ (hallucination)** คือกรณีที่ LLM ตอบอย่างมั่นใจในสิ่งที่ไม่เป็นความจริงหรือไม่ปรากฏในแหล่งข้อมูล เช่น การสร้างค่าผลตรวจทางห้องปฏิบัติการขึ้นเอง ซึ่งเป็นอันตรายในงานสุขภาพ มาตรการป้องกันเบื้องต้นคือการกำหนด **กฎการอ้างอิงแหล่งข้อมูล (grounding rule)** ในพรอมต์ ให้ตอบจากบันทึกเท่านั้น และหากไม่มีข้อมูล ให้ตอบว่า "Not documented" (ไม่มีการบันทึก)

In [ ]:
# ⭐ Exercise 3 — แบบฝึกหัดที่ 3: การทดสอบการสร้างข้อมูลเท็จ
# เป้าหมายทางคลินิก: ตรวจสอบว่าแบบจำลองสร้างข้อมูลที่ไม่ปรากฏในบันทึกหรือไม่
# บันทึกนี้ไม่มีผลตรวจคอเลสเตอรอล คำตอบที่ปลอดภัยจึงมีเพียง "Not documented"
# ความหมายของ SAFETY_RULE: "หากไม่มีข้อมูลในบันทึก ให้ตอบว่า 'Not documented' เท่านั้น ห้ามคาดเดา"
# ✏️ สิ่งที่ต้องทำ:
#   1. รันเซลล์โดยไม่แก้ไข และอ่านคำตอบทั้งสองแบบ (ไม่มีกฎความปลอดภัย / มีกฎความปลอดภัย)
#   2. เปลี่ยน `question` (ภายในเครื่องหมายอัญประกาศ) เป็นข้อมูลอื่นที่ไม่ปรากฏในบันทึก แล้วรันใหม่ เช่น
#      "What is the patient's weight?" (น้ำหนัก)  "What was the eGFR?"  "Which allergies does the patient have?" (ประวัติการแพ้)
#   3. ไม่บังคับ: แก้ไข SAFETY_RULE ให้เข้มงวดขึ้น
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   ทดสอบ "การสร้างข้อมูลเท็จ (hallucination)" โดยถามแบบจำลองถึงข้อมูลที่ไม่มีในบันทึก 2 แบบ
#     - แบบที่ 1: ไม่มีกฎความปลอดภัย (system มีเพียงบทบาท)
#     - แบบที่ 2: เพิ่ม "กฎการอ้างอิงแหล่งข้อมูล (grounding rule)" SAFETY_RULE ต่อท้าย system
#   แล้วแสดงคำตอบทั้งสองแบบคู่กัน เพื่อเปรียบเทียบว่ากฎช่วยลดการคาดเดาได้หรือไม่
#   ผลลัพธ์ที่ผู้เรียนจะเห็น: คำถาม และคำตอบ 2 แบบ (คำตอบที่ปลอดภัยควรเป็น "Not documented")
# ============================================================================

# กำหนดคำถามที่จะถามแบบจำลอง เก็บไว้ในตัวแปร question
#   ความหมาย: "ระดับ LDL คอเลสเตอรอลของผู้ป่วยเป็นเท่าใด" (ข้อมูลนี้ไม่มีในบันทึก)
#   - ผู้เรียนเปลี่ยนข้อความในเครื่องหมายอัญประกาศเป็นคำถามอื่นได้ตามข้อ 2 ด้านบน
question = "What was the patient's LDL cholesterol level?"      # ✏️ ทดลองคำถามอื่น

# กำหนดกฎความปลอดภัย เก็บไว้ในตัวแปร SAFETY_RULE (ชื่อตัวพิมพ์ใหญ่ทั้งหมดนิยมใช้กับค่าคงที่)
#   ความหมาย: "หากไม่มีข้อมูลในบันทึก ให้ตอบว่า 'Not documented' เท่านั้น ห้ามคาดเดา"
#   - ข้อความนี้ใช้อัญประกาศคู่ " " ด้านนอก จึงใส่อัญประกาศเดี่ยว ' ' ด้านในได้
SAFETY_RULE = "If the information is not in the note, answer exactly 'Not documented'. Never guess."   # ✏️ ไม่บังคับ

# แบบที่ 1 (ไม่มีกฎความปลอดภัย): ถามแบบจำลองแล้วเก็บคำตอบใน answer_plain
#   - "You are a clinical assistant." : system = บทบาทเพียงอย่างเดียว
#   - f"Note:\n{note}\n\nQuestion: {question}" : user = บันทึก ตามด้วยบรรทัดว่าง แล้วจึงเป็นคำถาม
#       (\n = ขึ้นบรรทัดใหม่, \n\n = เว้น 1 บรรทัด; {note} และ {question} คือค่าที่แทรกลงไป)
#   - max_new_tokens=150 : จำกัดคำตอบไม่เกิน 150 โทเคน (คำตอบสั้น ๆ ก็เพียงพอ)
answer_plain = chat("You are a clinical assistant.", f"Note:\n{note}\n\nQuestion: {question}", max_new_tokens=150)
# แบบที่ 2 (มีกฎความปลอดภัย): ถามคำถามเดิม แต่นำ SAFETY_RULE มาต่อท้าย system ด้วยเครื่องหมาย +
#   - "You are a clinical assistant. " + SAFETY_RULE : system = บทบาท + กฎความปลอดภัย
#   - user และ max_new_tokens เหมือนแบบที่ 1 ทุกประการ (เปลี่ยนเพียงปัจจัยเดียวเพื่อเปรียบเทียบอย่างเป็นธรรม)
answer_safe  = chat("You are a clinical assistant. " + SAFETY_RULE,
                    f"Note:\n{note}\n\nQuestion: {question}", max_new_tokens=150)
# แสดงคำถามที่ใช้ทดสอบ
print("❓", question)
# แสดงหัวข้อและคำตอบแบบที่ 1 (การใช้ + ต่อข้อความหัวข้อกับคำตอบเข้าด้วยกัน)
print("\n--- ไม่มีกฎความปลอดภัย ---\n" + answer_plain)
# แสดงหัวข้อและคำตอบแบบที่ 2
print("\n--- มีกฎความปลอดภัย ---\n" + answer_safe)

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ความหมายของกฎ: ตอบจากบันทึกเท่านั้น หากไม่มีข้อมูลให้ตอบ 'Not documented' ห้ามคาดเดาหรือใช้ค่าปกติทั่วไป
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เฉลยแบบฝึกหัดที่ 3: ใช้กฎความปลอดภัย (grounding rule) ที่เข้มงวดขึ้น แล้วทดสอบกับคำถาม 3 ข้อ
#   ซึ่งล้วนเป็นข้อมูลที่ไม่มีในบันทึก (LDL คอเลสเตอรอล, น้ำหนัก, eGFR) โดยใช้ for loop ถามทีละคำถาม
#   แต่ละคำถามจะถาม 2 แบบ (ไม่มีกฎ / มีกฎ) เพื่อเปรียบเทียบการสร้างข้อมูลเท็จ (hallucination)
#   ผลลัพธ์ที่ผู้เรียนจะเห็น: คำถาม 3 ข้อ แต่ละข้อมีคำตอบ 2 แบบ
# ============================================================================

# กฎความปลอดภัยฉบับเข้มงวด (ข้อความ 2 ชุดในวงเล็บต่อกันเป็นข้อความเดียว)
#   ความหมาย: "ตอบจากบันทึกเท่านั้น หากไม่มีข้อมูลในบันทึก ให้ตอบว่า 'Not documented' เท่านั้น
#             ห้ามคาดเดาหรือใช้ค่าที่พบโดยทั่วไป"
SAFETY_RULE = ("Answer ONLY from the note. If the information is not in the note, answer exactly "
               "'Not documented'. Never guess or use typical values.")
# วนซ้ำ (for loop) ผ่านคำถาม 3 ข้อใน list โดยแต่ละรอบเก็บคำถามไว้ในตัวแปร question
#   - คำถามที่ 1: "ระดับ LDL คอเลสเตอรอลเป็นเท่าใด"
#   - คำถามที่ 2: "ผู้ป่วยมีน้ำหนักเท่าใด"
#   - คำถามที่ 3: "ค่า eGFR (อัตราการกรองของไต) เป็นเท่าใด"
#   - คำสั่งที่ย่อหน้าเข้าไปใต้ for จะทำซ้ำทุกรอบ
for question in ["What was the patient's LDL cholesterol level?",
                 "What is the patient's weight?",
                 "What was the eGFR?"]:
    # แบบไม่มีกฎ: system มีเพียงบทบาท; user = บันทึก + คำถาม; จำกัดคำตอบ 150 โทเคน
    answer_plain = chat("You are a clinical assistant.", f"Note:\n{note}\n\nQuestion: {question}", max_new_tokens=150)
    # แบบมีกฎ: นำ SAFETY_RULE ต่อท้าย system ด้วย + ส่วน user และ max_new_tokens เหมือนเดิม
    answer_safe  = chat("You are a clinical assistant. " + SAFETY_RULE,
                        f"Note:\n{note}\n\nQuestion: {question}", max_new_tokens=150)
    # แสดงคำถามของรอบนี้
    print("❓", question)
    # แสดงคำตอบแบบไม่มีกฎความปลอดภัย
    print("--- ไม่มีกฎความปลอดภัย ---\n" + answer_plain)
    # แสดงคำตอบแบบมีกฎความปลอดภัย แล้วเว้นบรรทัด (\n) ก่อนคำถามถัดไป
    print("--- มีกฎความปลอดภัย ---\n" + answer_safe + "\n")

💡 **คำอธิบายผลลัพธ์**
หากไม่มีกฎ แบบจำลองอาจคาดเดาค่าที่พบบ่อย หรือให้คำแนะนำทั่วไปแทนการระบุว่าไม่มีข้อมูล กฎการอ้างอิงแหล่งข้อมูลมักช่วยลดปัญหานี้ได้ แต่พรอมต์ไม่ใช่หลักประกัน บุคลากรทางการแพทย์ยังต้องตรวจสอบคำตอบของ LLM ทุกครั้ง

✍️ **คำถามที่ 3:** สำหรับแต่ละคำถามที่ทดลอง แบบจำลองสร้างข้อมูลเท็จ (สร้างค่าหรือข้อเท็จจริงขึ้นเอง) หรือไม่ ทั้งในกรณีที่มีและไม่มีกฎความปลอดภัย โปรดยกตัวอย่างคำตอบประกอบ 1 ตัวอย่าง

✍️ **คำตอบ:**



✍️ **คำถามที่ 4:** จงอธิบายใน 2 ประโยคว่า เหตุใดการ **อ้างอิงแหล่งข้อมูล (grounding)** ที่เชื่อถือได้ (ปฏิบัติการถัดไป: retrieval-augmented generation, RAG) จึงมีความสำคัญต่อการใช้งานทางคลินิก และโปรดระบุมาตรการป้องกัน 1 ประการที่ผู้เรียนจะกำหนดก่อนนำ LLM ไปใช้กับบันทึกจริงในหน่วยงาน

✍️ **คำตอบ:**



## 🏆 โจทย์ท้าทายบนตารางอันดับ (Leaderboard) — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้

> **ส่วนนี้ไม่บังคับ และไม่นับเป็นส่วนหนึ่งของชิ้นงานที่ต้องส่ง** โค้ดตั้งต้น (baseline) ด้านล่างรันได้ทันทีและสร้างไฟล์ส่งผลที่ถูกต้อง ผู้เรียนทุกคนจึงสามารถส่งผลได้

ให้รันโปรแกรมสกัดข้อมูลกับ **บันทึกทางคลินิกสังเคราะห์ 20 ฉบับ** ของรายวิชา แล้วส่งค่า `age`, `sex`, `conditions` และ `medications` ของแต่ละบันทึก **ตัวชี้วัด: ค่าเฉลี่ยของ age accuracy, sex accuracy, conditions F1 และ medications F1** (F1 พิจารณาทั้งรายการที่ตกหล่นและรายการเกินที่ไม่ถูกต้อง ระบบยอมรับคำพ้องความหมาย เช่น *HTN* / *hypertension* หรือ *T2DM* / *type 2 diabetes* และเปรียบเทียบชื่อยาโดยไม่พิจารณาขนาดยา)

ไฟล์ส่งผล: `lab08_submission.json` — JSON object ที่มีคีย์เป็นรหัสบันทึก เช่น `{"N01": {"age": 57, "sex": "M", "conditions": [...], "medications": [...]}}` หากต้องการส่งผล ให้ระบุ `NAME` และ `JOIN_CODE` ในเซลล์การตั้งค่า

> **หลักความซื่อสัตย์ทางวิชาการ (honor code):** ผลการทำนายต้องได้จากพรอมต์ แบบจำลอง หรือโค้ดของผู้เรียนเท่านั้น ห้ามอ่านบันทึกชุดทดสอบแล้วพิมพ์คำตอบด้วยตนเอง งานนี้เป็นงานเดี่ยว

ขั้นที่ 1 — พรอมต์ตั้งต้น (baseline) ทดสอบกับบันทึกฝึกหัด 3 ฉบับที่ทราบคำตอบ (ไม่ใช่ชุดทดสอบ) พร้อมคะแนนโดยประมาณ

In [ ]:
# 🏆 โปรแกรมสกัดข้อมูลตั้งต้น และคะแนนโดยประมาณบนบันทึกฝึกหัด 3 ฉบับ
# ความหมายของพรอมต์: ตอบเป็น JSON เท่านั้น มีคีย์ age (จำนวนเต็ม), sex ('M'/'F'), conditions (รายการโรค), medications (รายการชื่อยา)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (ส่วนโจทย์ท้าทาย — ไม่บังคับ)
#   เซลล์นี้สร้าง "โปรแกรมสกัดข้อมูลตั้งต้น (baseline)" และ "ตัวให้คะแนนโดยประมาณ" ได้แก่
#     1) EXTRACT_SYSTEM : พรอมต์ system ตั้งต้นสำหรับสกัด age / sex / conditions / medications
#     2) extract_note(...) : สกัดข้อมูลจากบันทึก 1 ฉบับ แล้วจัดรูปแบบให้ตรงกับที่ตารางอันดับกำหนด
#     3) DEV_NOTES : บันทึกฝึกหัด (development set) 3 ฉบับ พร้อม "เฉลย (gold)" ที่ถูกต้อง
#     4) ตัวช่วยปรับคำให้เป็นมาตรฐาน (_norm_cond, _norm_med) และตัวคำนวณ F1 (_f1)
#     5) rough_score(...) : คำนวณคะแนนโดยประมาณ = ค่าเฉลี่ยของ age accuracy, sex accuracy,
#        conditions F1 และ medications F1 (เหมือนตัวชี้วัดของตารางอันดับ แต่ยืดหยุ่นน้อยกว่า)
#   ผลลัพธ์ที่ผู้เรียนจะเห็น: ผลการทำนายเทียบกับเฉลยของบันทึกฝึกหัดทั้ง 3 ฉบับ และคะแนนโดยประมาณของ baseline
#   (ความแม่นยำ (accuracy) = สัดส่วนที่ตอบถูก; F1 = ค่าที่รวมทั้งการตกหล่นและการตอบเกิน มีค่า 0 ถึง 1 ยิ่งสูงยิ่งดี)
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: พรอมต์ตั้งต้น และฟังก์ชันสกัดข้อมูลจากบันทึก 1 ฉบับ
# ----------------------------------------------------------------------------
# พรอมต์ system ตั้งต้น เก็บในตัวแปร EXTRACT_SYSTEM (ข้อความ 2 ชุดในวงเล็บต่อกันเป็นข้อความเดียว)
#   ความหมาย: "คุณคือผู้สกัดข้อมูลทางคลินิก ให้ตอบเป็น JSON ที่ถูกต้องเท่านั้น โดยมีคีย์ age (จำนวนเต็ม),
#             sex ('M' หรือ 'F'), conditions (รายการการวินิจฉัย), medications (รายการชื่อยา)"
EXTRACT_SYSTEM = ("You are a clinical information extractor. Respond with ONLY valid JSON with keys "
                  "age (integer), sex ('M' or 'F'), conditions (list of diagnoses), medications (list of drug names).")

# สร้างฟังก์ชัน extract_note สำหรับสกัดข้อมูลจากบันทึก 1 ฉบับ
#   - text   : ข้อความบันทึกทางคลินิก
#   - system : พรอมต์ system ที่จะใช้ (เช่น EXTRACT_SYSTEM)
#   - examples=None : (ไม่บังคับ) ตัวอย่าง few-shot เป็นรายการคู่ (ข้อความบันทึก, คำตอบแบบ dictionary)
#   - max_new_tokens=200 : ความยาวคำตอบสูงสุด 200 โทเคน
def extract_note(text, system, examples=None, max_new_tokens=200):
    """รันแบบจำลองกับบันทึก 1 ฉบับ; examples = (ไม่บังคับ) รายการตัวอย่าง few-shot คู่ (ข้อความบันทึก, คำตอบแบบ dict)"""
    # แปลงตัวอย่าง few-shot ให้อยู่ในรูปแบบที่ chat(...) ต้องการ คือคู่ (ข้อความคำขอ, ข้อความคำตอบ) (list comprehension)
    #   - for n, a in (examples or []) : วนแต่ละคู่ โดย n = ข้อความบันทึก, a = คำตอบแบบ dictionary
    #                                    (หาก examples เป็น None จะใช้ list ว่าง คือไม่มีตัวอย่าง)
    #   - f"Note:\n{n}" : สร้างคำขอตัวอย่างในรูปแบบเดียวกับคำขอจริง ("Note:" ขึ้นบรรทัดใหม่แล้วตามด้วยบันทึก)
    #   - json.dumps(a) : แปลงคำตอบ dictionary เป็นข้อความ JSON (ให้แบบจำลองเห็นรูปแบบคำตอบที่ถูกต้อง)
    shots = [(f"Note:\n{n}", json.dumps(a)) for n, a in (examples or [])]
    # ถามแบบจำลอง แล้วเก็บคำตอบดิบไว้ใน raw
    #   - system : พรอมต์ system ที่รับเข้ามา
    #   - f"Note:\n{text}" : คำขอ = "Note:" ตามด้วยข้อความบันทึก
    #   - max_new_tokens=max_new_tokens : ส่งต่อค่าความยาวสูงสุดที่รับเข้ามา
    #   - examples=shots : ตัวอย่าง few-shot ที่แปลงรูปแบบแล้ว
    raw = chat(system, f"Note:\n{text}", max_new_tokens=max_new_tokens, examples=shots)
    # แปลงคำตอบดิบเป็น dictionary แล้วจัดรูปแบบมาตรฐาน จากนั้นส่งกลับ (ทำงานจากในออกนอก)
    #   1) parse_json_reply(raw, quiet=True) : ดึง JSON จากคำตอบ โดยไม่แสดงคำเตือนหากหาไม่พบ (quiet=True)
    #   2) clean_record(...) : คงไว้เฉพาะ age / sex / conditions / medications ในรูปแบบที่ตารางอันดับกำหนด
    return clean_record(parse_json_reply(raw, quiet=True))

# ----------------------------------------------------------------------------
# ส่วนที่ 2: บันทึกฝึกหัด 3 ฉบับพร้อมเฉลย (DEV_NOTES)
#   โครงสร้าง: list ของ dictionary โดยแต่ละรายการมี 2 คีย์
#     - "note" : ข้อความบันทึก (ข้อความหลายชุดในวงเล็บที่วางติดกันจะต่อกันเป็นข้อความเดียว)
#     - "gold" : เฉลยที่ถูกต้อง (dictionary ที่มี age, sex, conditions, medications)
# ----------------------------------------------------------------------------
DEV_NOTES = [
    # บันทึกที่ 1: ชายอายุ 72 ปี เป็นความดันโลหิตสูง (HTN), โรคปอดอุดกั้นเรื้อรัง (COPD) และหัวใจห้องบนสั่นพลิ้ว
    #   (atrial fibrillation) มีเท้าบวม 2 สัปดาห์; ยา amlodipine, apixaban, tiotropium (ยาพ่น)
    #   - เฉลย: อายุ 72, เพศ M, โรค 3 รายการ, ยา 3 รายการ (เฉพาะชื่อยา ไม่รวมขนาดยา)
    #   - สังเกต: อาการ "ankle swelling" และ "No fever" ไม่นับเป็นโรค
    {"note": "72M with HTN, COPD and atrial fibrillation presents for follow-up. Reports ankle swelling x2 weeks. "
             "BP 146/88, HR 78. Medications: amlodipine 5mg daily; apixaban 5 mg BID; tiotropium inhaler daily. "
             "No fever. Plan: recheck BP in 4 weeks.",
     "gold": {"age": 72, "sex": "M", "conditions": ["hypertension", "copd", "atrial fibrillation"],
              "medications": ["amlodipine", "apixaban", "tiotropium"]}},
    # บันทึกที่ 2: หญิงอายุ 39 ปี เป็นหอบหืด (asthma) และภาวะไทรอยด์ทำงานต่ำ (hypothyroidism)
    #   มีหายใจมีเสียงวี้ดกลางคืน 1 สัปดาห์; ยา levothyroxine และ montelukast
    #   - เฉลย: อายุ 39, เพศ F, โรค 2 รายการ, ยา 2 รายการ
    #   - สังเกต: "Denies chest pain" (ปฏิเสธเจ็บหน้าอก) ไม่นับเป็นโรค
    {"note": "Patient is a 39 year old female with asthma and hypothyroidism, seen today for review. Reports night-time "
             "wheeze x1 week. BP 118/70, HR 84. Current meds include levothyroxine 75 mcg daily and montelukast 10 mg "
             "at night. Denies chest pain. Plan: review inhaler technique, TSH in 6 weeks.",
     "gold": {"age": 39, "sex": "F", "conditions": ["asthma", "hypothyroidism"],
              "medications": ["levothyroxine", "montelukast"]}},
    # บันทึกที่ 3: ชายอายุ 61 ปี เป็นโรคไตเรื้อรังระยะที่ 3 (CKD stage 3) และเบาหวานชนิดที่ 2
    #   มีอ่อนเพลีย 1 เดือน; ยา metformin, empagliflozin, atorvastatin
    #   - เฉลย: อายุ 61, เพศ M, โรค 2 รายการ, ยา 3 รายการ
    {"note": "61-year-old man with a history of CKD stage 3 and type 2 diabetes presents for medication review. Reports "
             "fatigue x1 month. BP 138/84, HR 72. HbA1c 7.9%. Medications: metformin 500mg BID; empagliflozin 10 mg "
             "daily; atorvastatin 40mg at bedtime. No chest pain. Plan: continue current regimen, recheck eGFR in 3 months.",
     "gold": {"age": 61, "sex": "M", "conditions": ["chronic kidney disease", "type 2 diabetes"],
              "medications": ["metformin", "empagliflozin", "atorvastatin"]}},
]

# ----------------------------------------------------------------------------
# ส่วนที่ 3: ตัวช่วยปรับชื่อโรค/ยาให้เป็นมาตรฐาน ก่อนนำไปเทียบกับเฉลย
# ----------------------------------------------------------------------------
# ตารางคำพ้องความหมายและคำย่อของโรค (dictionary แบบ "คำที่พบ: ชื่อมาตรฐาน")
#   เช่น "htn" และ "high blood pressure" -> "hypertension", "afib" -> "atrial fibrillation",
#   "ckd" -> "chronic kidney disease", "t2dm" -> "type 2 diabetes"
#   (dictionary เขียนต่อกันหลายบรรทัดได้ เพราะอยู่ภายในวงเล็บปีกกา { })
_ALIASES = {"htn": "hypertension", "high blood pressure": "hypertension", "af": "atrial fibrillation",
            "afib": "atrial fibrillation", "chronic obstructive pulmonary disease": "copd",
            "ckd": "chronic kidney disease", "t2dm": "type 2 diabetes", "dm2": "type 2 diabetes",
            "type 2 diabetes mellitus": "type 2 diabetes", "diabetes mellitus type 2": "type 2 diabetes"}

# ฟังก์ชัน _norm_cond : ปรับชื่อโรค c 1 รายการให้เป็นรูปแบบมาตรฐาน
def _norm_cond(c):
    # ทำความสะอาดชื่อโรค (ทำทีละขั้นจากในออกนอก)
    #   1) str(c).lower() : แปลงเป็นข้อความตัวพิมพ์เล็ก เช่น "CKD stage 3" -> "ckd stage 3"
    #   2) re.sub(รูปแบบ, "", ...) : ลบส่วนที่ตรงรูปแบบออก โดยรูปแบบมี 2 ทางเลือกคั่นด้วย | (หรือ)
    #        * \s*\(.*?\)       : ข้อความในวงเล็บ (พร้อมช่องว่างด้านหน้า) เช่น " (HTN)"
    #        * \bstage \w+\b    : คำว่า "stage" ตามด้วยคำ 1 คำ เช่น "stage 3"
    #   3) .strip(" .") : ตัดช่องว่างและจุดที่หัวท้ายข้อความ
    c = re.sub(r"\s*\(.*?\)|\bstage \w+\b", "", str(c).lower()).strip(" .")
    # แปลงคำย่อเป็นชื่อมาตรฐานด้วยตาราง _ALIASES
    #   - .get(c, c) : หากพบ c ในตารางให้คืนชื่อมาตรฐาน หากไม่พบให้คืน c ตามเดิม
    return _ALIASES.get(c, c)

# ฟังก์ชัน _norm_med : ปรับชื่อยา m ให้เหลือเฉพาะคำแรก (ตัดขนาดยาและความถี่ออก)
def _norm_med(m):
    # แปลงเป็นตัวพิมพ์เล็ก แล้วแยกเป็นคำ ๆ ตามช่องว่างด้วย .split()
    #   เช่น "Metformin 500mg BID" -> ["metformin", "500mg", "bid"]
    words = str(m).lower().split()
    # คืนคำแรก words[0] (ดัชนี 0 = ตัวแรก) หากมีคำอยู่ มิฉะนั้น (ข้อความว่าง) คืนข้อความว่าง ""
    return words[0] if words else ""

# ----------------------------------------------------------------------------
# ส่วนที่ 4: ฟังก์ชัน _f1 คำนวณค่า F1 รวมทุกบันทึก (micro-F1)
#   - pairs : list ของคู่ (รายการเฉลย, รายการที่ทำนาย) ของแต่ละบันทึก
#   สูตร: F1 = 2 × จำนวนที่ตรงกัน / (จำนวนที่ทำนายทั้งหมด + จำนวนเฉลยทั้งหมด)
#   ค่า F1 จะลดลงทั้งเมื่อตกหล่นรายการที่ถูกต้อง และเมื่อตอบรายการเกินที่ไม่ถูกต้อง
# ----------------------------------------------------------------------------
def _f1(pairs):
    # นับจำนวนรายการที่ทำนายถูก (true positive, tp) รวมทุกบันทึก
    #   - for g, p in pairs : วนทุกคู่ โดย g = รายการเฉลย, p = รายการที่ทำนาย
    #   - set(...) : แปลง list เป็นเซต (ตัดรายการซ้ำ)
    #   - set(g) & set(p) : รายการที่อยู่ในทั้งสองเซต (ส่วนที่ตรงกัน)
    #   - len(...) : นับจำนวน แล้ว sum(...) รวมทุกบันทึก
    tp = sum(len(set(g) & set(p)) for g, p in pairs)
    # นับจำนวนรายการที่ทำนายทั้งหมด (n_pred) และจำนวนเฉลยทั้งหมด (n_gold) ในบรรทัดเดียว
    #   - การเขียน a, b = ค่า1, ค่า2 คือการกำหนดค่าให้ 2 ตัวแปรพร้อมกัน
    #   - for _, p in pairs : ใช้เฉพาะรายการที่ทำนาย (_ = ส่วนที่ไม่ใช้)
    #   - for g, _ in pairs : ใช้เฉพาะรายการเฉลย
    n_pred, n_gold = sum(len(set(p)) for _, p in pairs), sum(len(set(g)) for g, _ in pairs)
    # คำนวณ F1 = 2 × tp / (n_pred + n_gold)
    #   - if n_pred + n_gold else 1.0 : หากทั้งทำนายและเฉลยว่างทั้งคู่ (ผลรวมเป็น 0) ถือว่าถูกต้องสมบูรณ์ = 1.0
    #     (ป้องกันการหารด้วยศูนย์)
    return 2 * tp / (n_pred + n_gold) if n_pred + n_gold else 1.0

# ----------------------------------------------------------------------------
# ส่วนที่ 5: ฟังก์ชัน rough_score คำนวณคะแนนโดยประมาณบน DEV_NOTES
#   - system : พรอมต์ system ที่ต้องการทดสอบ
#   - examples=None : (ไม่บังคับ) ตัวอย่าง few-shot
#   - show=True : แสดงผลการทำนายเทียบกับเฉลยของแต่ละบันทึก (False = ไม่แสดง)
#   ผลลัพธ์: คะแนนเฉลี่ยของ 4 ตัวชี้วัด (ตัวเลขระหว่าง 0 ถึง 1)
# ----------------------------------------------------------------------------
def rough_score(system, examples=None, show=True):
    """ประมาณค่าตัวชี้วัดของตารางอันดับบน DEV_NOTES (ระบบให้คะแนนจริงรองรับคำพ้องความหมายมากกว่านี้)"""
    # สกัดข้อมูลจากบันทึกฝึกหัดทุกฉบับ (list comprehension) แล้วเก็บผลการทำนายใน preds
    #   - for d in DEV_NOTES : วนบันทึกทีละรายการ
    #   - d["note"] : ดึงข้อความบันทึกด้วยการระบุคีย์ในวงเล็บเหลี่ยม
    #   - extract_note(d["note"], system, examples) : สกัดข้อมูลด้วยพรอมต์และตัวอย่างที่กำหนด
    preds = [extract_note(d["note"], system, examples) for d in DEV_NOTES]
    # ดึงเฉลยของทุกบันทึกเก็บไว้ใน golds (ลำดับเดียวกับ preds)
    golds = [d["gold"] for d in DEV_NOTES]
    # หาก show เป็นจริง ให้แสดงผลการทำนายเทียบกับเฉลย
    if show:
        # วนผลการทำนาย p และเฉลย g ไปพร้อมกันเป็นคู่ ๆ
        #   - zip(preds, golds) : จับคู่รายการลำดับเดียวกันจาก 2 list (ตัวที่ 1 กับตัวที่ 1, ...)
        for p, g in zip(preds, golds):
            # แสดงผลการทำนายและเฉลยของบันทึกนี้ (\n = ขึ้นบรรทัดใหม่ให้เฉลยอยู่บรรทัดถัดไป)
            print("ผลการทำนาย:", p, "\nเฉลย      :", g, "\n")
    # ความแม่นยำของอายุ (age accuracy) = จำนวนบันทึกที่อายุตรงกับเฉลย ÷ จำนวนบันทึกทั้งหมด
    #   - p["age"] == g["age"] : เปรียบเทียบว่าเท่ากันหรือไม่ ได้ True/False
    #   - sum(...) : ในการบวก Python นับ True เป็น 1 และ False เป็น 0 จึงได้จำนวนที่ตรงกัน
    #   - len(golds) : จำนวนบันทึกทั้งหมด (3)
    age = sum(p["age"] == g["age"] for p, g in zip(preds, golds)) / len(golds)
    # ความแม่นยำของเพศ (sex accuracy) คำนวณแบบเดียวกัน
    sex = sum(p["sex"] == g["sex"] for p, g in zip(preds, golds)) / len(golds)
    # ค่า F1 ของโรค (conditions F1)
    #   - สร้างคู่ (เฉลยที่ปรับมาตรฐานแล้ว, คำทำนายที่ปรับมาตรฐานแล้ว) ของแต่ละบันทึก
    #   - [_norm_cond(c) for c in g["conditions"]] : ปรับชื่อโรคในเฉลยทุกตัวให้เป็นมาตรฐาน
    #   - [_norm_cond(c) for c in p["conditions"]] : ปรับชื่อโรคที่ทำนายทุกตัวให้เป็นมาตรฐาน
    #   - แล้วส่ง list ของคู่ทั้งหมดให้ _f1(...) คำนวณ
    cond = _f1([([_norm_cond(c) for c in g["conditions"]], [_norm_cond(c) for c in p["conditions"]]) for p, g in zip(preds, golds)])
    # ค่า F1 ของยา (medications F1) คำนวณแบบเดียวกัน แต่ใช้ _norm_med (เหลือเฉพาะชื่อยาคำแรก)
    med = _f1([([_norm_med(m) for m in g["medications"]], [_norm_med(m) for m in p["medications"]]) for p, g in zip(preds, golds)])
    # แสดงคะแนนแต่ละตัวชี้วัดและคะแนนเฉลี่ย (f-string 2 ชุดวางติดกันจึงต่อเป็นข้อความเดียว)
    #   - {age:.2f} : แสดงค่า age เป็นทศนิยม 2 ตำแหน่ง (:.2f = รูปแบบทศนิยม 2 ตำแหน่ง)
    #   - {(age + sex + cond + med) / 4:.3f} : คำนวณค่าเฉลี่ยของ 4 ตัวชี้วัด แล้วแสดงทศนิยม 3 ตำแหน่ง
    print(f"age acc {age:.2f} | sex acc {sex:.2f} | conditions F1 {cond:.2f} | medications F1 {med:.2f} "
          f"| คะแนนโดยประมาณ {(age + sex + cond + med) / 4:.3f}")
    # ส่งคะแนนเฉลี่ยกลับ เพื่อนำไปเก็บในตัวแปรและเปรียบเทียบภายหลัง
    return (age + sex + cond + med) / 4

# ----------------------------------------------------------------------------
# ส่วนที่ 6: คำนวณคะแนนของพรอมต์ตั้งต้น (baseline)
# ----------------------------------------------------------------------------
# รัน rough_score(...) กับพรอมต์ EXTRACT_SYSTEM (ไม่มีตัวอย่าง few-shot) แล้วเก็บคะแนนไว้ใน baseline_score
#   (ใช้เปรียบเทียบกับพรอมต์ที่ผู้เรียนปรับปรุงในเซลล์ถัดไป)
baseline_score = rough_score(EXTRACT_SYSTEM)

**แนวทางการปรับปรุงให้ได้คะแนนสูงกว่า baseline**

- ระบุโครงสร้าง JSON (schema) ในพรอมต์อย่างชัดเจน พร้อมตัวอย่างรูปแบบผลลัพธ์ที่ต้องการ
- เพิ่ม **ตัวอย่างแบบ few-shot** 1–2 ตัวอย่าง (บันทึกพร้อมคำตอบที่ถูกต้อง) ดังที่แสดงใน `MY_EXAMPLES` ด้านล่าง
- กำหนดขอบเขตของโรคให้ชัดเจน: นับเฉพาะการวินิจฉัยจากประวัติ ("with ...", "history of ...") **ไม่นับ** อาการนำ ("Reports knee pain") หรือสิ่งที่ผู้ป่วยปฏิเสธ ("No chest pain", "Denies dyspnea")
- กำหนดว่า "copy diagnoses exactly as written" (ระบบให้คะแนนยอมรับคำพ้องความหมายและคำย่อ) และ "drug names only, without dose or frequency"
- กำหนด `max_new_tokens` ให้เหมาะสม และใช้ Python ประมวลผลต่อ (เช่น ตัดรายการซ้ำ หรือใช้ regex ระบุอายุและเพศ)
- ใช้แบบจำลองแบบเปิดที่มีขนาดใหญ่ขึ้น เช่น `Qwen2.5-3B-Instruct` ซึ่งทำงานบน T4 แบบ fp16 ได้ (เปลี่ยนค่า `MODEL` แล้วโหลดใหม่)

ขั้นที่ 2 — พรอมต์ที่ผู้เรียนปรับปรุง (ไม่บังคับ; แก้ไขข้อความแล้วรันใหม่เพื่อเปรียบเทียบกับคะแนน baseline)

In [ ]:
# 🏆 ไม่บังคับ — พรอมต์ที่ผู้เรียนปรับปรุง: แก้ไขข้อความภายในเครื่องหมายอัญประกาศ รันใหม่ แล้วเปรียบเทียบคะแนนโดยประมาณ
# ความหมายของข้อความที่เพิ่ม: conditions = โรคที่ผู้ป่วยเป็น (จากประวัติ) คัดลอกตามที่บันทึก ไม่รวมอาการนำหรือสิ่งที่ปฏิเสธ;
# medications = ชื่อยาเท่านั้น ไม่รวมขนาดยาและความถี่
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (ส่วนโจทย์ท้าทาย — ไม่บังคับ)
#   ทดลองปรับปรุงพรอมต์ด้วย 2 เทคนิค แล้ววัดคะแนนโดยประมาณบนบันทึกฝึกหัดเทียบกับ baseline
#     1) เพิ่มคำนิยามที่ชัดเจนต่อท้ายพรอมต์ตั้งต้น (MY_SYSTEM)
#     2) เพิ่มตัวอย่างแบบ few-shot (MY_EXAMPLES) คือคู่ (บันทึก, คำตอบที่ถูกต้อง) ให้แบบจำลองเห็นรูปแบบ
#   ผลลัพธ์ที่ผู้เรียนจะเห็น: ผลการทำนายเทียบกับเฉลย คะแนนโดยประมาณ และการเปรียบเทียบ baseline → พรอมต์ที่ปรับปรุง
# ============================================================================

# สร้างพรอมต์ใหม่ MY_SYSTEM โดยนำ EXTRACT_SYSTEM (จากเซลล์ก่อนหน้า) มาต่อด้วยข้อความเพิ่มเติม (เครื่องหมาย +)
#   ความหมายของข้อความที่เพิ่ม: "Conditions = การวินิจฉัยที่ผู้ป่วยเป็นอยู่ (จากประวัติ) คัดลอกตามที่เขียน;
#   ห้ามรวมอาการนำ หรือสิ่งที่ตรวจไม่พบ/ปฏิเสธ เช่น 'No chest pain'. Medications = ชื่อยาเท่านั้น ไม่รวมขนาดยาหรือความถี่"
#   - ข้อความ 2 ชุดในวงเล็บที่วางติดกันจะต่อกันเป็นข้อความเดียว (ข้อความเริ่มด้วยช่องว่าง เพื่อเว้นวรรคจากประโยคเดิม)
MY_SYSTEM = EXTRACT_SYSTEM + (
    " Conditions = diagnoses the patient HAS (from the history), copied as written; do NOT include presenting"
    " symptoms or negated findings such as 'No chest pain'. Medications = drug names only, without dose or frequency.")

# รายการตัวอย่าง few-shot เก็บใน MY_EXAMPLES (list ของคู่ (ข้อความบันทึก, คำตอบที่ถูกต้องแบบ dictionary))
#   - ตัวอย่างที่มีอยู่ 1 คู่: ใช้บันทึก note (ผู้ป่วยชาย 57 ปี จากแบบฝึกหัดก่อนหน้า) พร้อมคำตอบที่ถูกต้อง
#       อายุ 57, เพศ M, โรค ["T2DM"], ยา ["metformin"]
#   - (a, b) คือ tuple (คู่ค่า) ; จุลภาคท้ายรายการทำให้เพิ่มตัวอย่างอื่นต่อได้ง่าย
MY_EXAMPLES = [   # ตัวอย่างแบบ few-shot: (ข้อความบันทึก, คำตอบที่ถูกต้อง)
    (note, {"age": 57, "sex": "M", "conditions": ["T2DM"], "medications": ["metformin"]}),
]

# คำนวณคะแนนโดยประมาณของพรอมต์ที่ปรับปรุง แล้วเก็บไว้ใน my_score
#   - MY_SYSTEM   : พรอมต์ system ที่ปรับปรุงแล้ว
#   - MY_EXAMPLES : ตัวอย่าง few-shot
my_score = rough_score(MY_SYSTEM, MY_EXAMPLES)
# แสดงการเปรียบเทียบคะแนน baseline กับพรอมต์ที่ปรับปรุง
#   - {baseline_score:.3f} และ {my_score:.3f} : แสดงคะแนนเป็นทศนิยม 3 ตำแหน่ง
#   - \n ด้านหน้า : เว้นบรรทัดก่อนแสดง
print(f"\nbaseline {baseline_score:.3f}  →  พรอมต์ที่ปรับปรุง {my_score:.3f}")

ขั้นที่ 3 — รันกับบันทึก 20 ฉบับของรายวิชา บันทึกไฟล์ `lab08_submission.json` และส่งผล (ใช้เวลาประมาณ 2–4 นาทีบน GPU T4)

In [ ]:
# 🏆 สกัดข้อมูลจากบันทึกชุดทดสอบทั้ง 20 ฉบับ และส่งผล
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (ส่วนโจทย์ท้าทาย — ไม่บังคับ)
#   1) เลือกพรอมต์และตัวอย่าง few-shot ที่จะใช้จริง (FINAL_SYSTEM, FINAL_EXAMPLES)
#   2) ดาวน์โหลดบันทึกทางคลินิกสังเคราะห์ 20 ฉบับของรายวิชา แล้วสกัด age / sex / conditions / medications ทีละฉบับ
#   3) บันทึกผลทั้งหมดเป็นไฟล์ lab08_submission.json
#   4) หากกรอก NAME และ JOIN_CODE ในเซลล์การตั้งค่าแล้ว จะส่งไฟล์ไปยังตารางอันดับโดยอัตโนมัติ
#   ผลลัพธ์ที่ผู้เรียนจะเห็น: ความคืบหน้าทีละบันทึก (1/20 ... 20/20) ข้อความยืนยันการบันทึกไฟล์
#   และคะแนน/อันดับ (หากส่งผล) — ใช้เวลาประมาณ 2–4 นาทีบน GPU T4
# ============================================================================

# กำหนดพรอมต์และตัวอย่างที่จะใช้จริง 2 ตัวแปรพร้อมกัน (a, b = ค่า1, ค่า2)
#   - FINAL_SYSTEM = EXTRACT_SYSTEM : ใช้พรอมต์ตั้งต้น
#   - FINAL_EXAMPLES = None : ไม่ใช้ตัวอย่าง few-shot
#   - หมายเหตุท้ายบรรทัด: หากพรอมต์ที่ปรับปรุงได้คะแนนสูงกว่า ให้เปลี่ยนเป็น MY_SYSTEM, MY_EXAMPLES
FINAL_SYSTEM, FINAL_EXAMPLES = EXTRACT_SYSTEM, None   # ✏️ เปลี่ยนเป็น MY_SYSTEM, MY_EXAMPLES หากพรอมต์ที่ปรับปรุงได้คะแนนสูงกว่า

# ดาวน์โหลดและอ่านไฟล์บันทึกชุดทดสอบ แล้วเก็บไว้ในตัวแปร notes (ทำงานจากในออกนอก)
#   1) lab_data("notes/notes.json") : ฟังก์ชันจากเซลล์การตั้งค่า ดาวน์โหลดไฟล์ notes.json และคืนตำแหน่งไฟล์ในเครื่อง
#   2) open(...) : เปิดไฟล์เพื่ออ่าน
#   3) json.load(...) : อ่านเนื้อหา JSON ในไฟล์แล้วแปลงเป็นข้อมูล Python
#      (ในที่นี้คือ list ของ dictionary ที่มีคีย์ "id" และ "note")
notes = json.load(open(lab_data("notes/notes.json")))
# สร้าง dictionary ว่าง submission สำหรับเก็บผล โดยจะใช้รหัสบันทึก (id) เป็นคีย์
submission = {}
# วนบันทึกทุกฉบับ พร้อมเลขลำดับ
#   - enumerate(notes, 1) : ให้ทั้งเลขลำดับ i และรายการ n โดยเริ่มนับที่ 1 (ค่าเริ่มต้นปกติคือ 0)
for i, n in enumerate(notes, 1):
    # สกัดข้อมูลจากบันทึกนี้ แล้วเก็บใน submission โดยใช้ n["id"] เป็นคีย์
    #   - n["note"] : ข้อความบันทึก
    #   - FINAL_SYSTEM, FINAL_EXAMPLES : พรอมต์และตัวอย่างที่เลือกไว้ด้านบน
    submission[n["id"]] = extract_note(n["note"], FINAL_SYSTEM, FINAL_EXAMPLES)
    # แสดงความคืบหน้า เช่น " 1/20 note_01: {...}"
    #   - {i:2d} : แสดง i เป็นจำนวนเต็มความกว้าง 2 ช่อง (d = จำนวนเต็ม) เพื่อให้ตัวเลขเรียงตรงกัน
    #   - {len(notes)} : จำนวนบันทึกทั้งหมด
    #   - {n['id']} : รหัสบันทึก (ภายใน f-string ที่ใช้ " " ด้านนอก จึงใช้ ' ' ด้านใน)
    #   - {submission[n['id']]} : ผลที่สกัดได้ของบันทึกนี้
    print(f"{i:2d}/{len(notes)} {n['id']}: {submission[n['id']]}")

# บันทึกผลทั้งหมดลงไฟล์ JSON
#   - json.dump(ข้อมูล, ไฟล์, ...) : เขียนข้อมูล Python ลงไฟล์ในรูปแบบ JSON
#   - submission : ข้อมูลที่ต้องการบันทึก
#   - open("lab08_submission.json", "w") : เปิด (หรือสร้าง) ไฟล์ชื่อนี้ในโหมดเขียน ("w" = write เขียนทับของเดิม)
#   - indent=1 : ย่อหน้าแต่ละระดับ 1 ช่องว่าง ให้อ่านไฟล์ได้ง่าย
json.dump(submission, open("lab08_submission.json", "w"), indent=1)
# แสดงข้อความยืนยัน พร้อมจำนวนบันทึกที่บันทึกลงไฟล์ (len(submission))
print("\nบันทึกไฟล์ lab08_submission.json เรียบร้อย จำนวน", len(submission), "บันทึก")
# ตรวจว่ากรอกทั้ง NAME และ JOIN_CODE ในเซลล์การตั้งค่าแล้วหรือไม่ (ข้อความว่างถือเป็นเท็จ)
if NAME and JOIN_CODE:
    # ส่งไฟล์ไปยังตารางอันดับด้วยฟังก์ชันจากเซลล์การตั้งค่า และแสดงคะแนนพร้อมอันดับ
    #   - "lab08_submission.json" : ชื่อไฟล์ผลที่จะส่ง
    submit_to_leaderboard("lab08_submission.json")
# กรณียังไม่ได้กรอก NAME หรือ JOIN_CODE
else:
    # แสดงคำแนะนำวิธีส่งผล
    print("หากต้องการส่งผล โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่า รันเซลล์ดังกล่าว แล้วรันเซลล์นี้อีกครั้ง")

## ชิ้นงานที่ต้องส่ง

ปฏิบัติการนี้เป็น **งานเดี่ยว** ให้ส่งชิ้นงานตามช่องทางที่รายวิชากำหนด ดังนี้

1. โน้ตบุ๊กนี้ซึ่ง **รันครบทุกเซลล์ตามลำดับจากบนลงล่าง** (แสดงผลลัพธ์ของสรุปที่ปรับปรุงแล้ว การสกัดข้อมูล JSON ที่แปลงได้สำเร็จ และการทดสอบการสร้างข้อมูลเท็จ)
2. คำตอบ ✍️ ของคำถามที่ 1–4 (รวมถึงข้อสรุป 2 ประโยคเกี่ยวกับความสำคัญของการอ้างอิงแหล่งข้อมูล)

*ไม่บังคับ:* ผู้เรียนที่เข้าร่วมโจทย์ท้าทายบน 🏆 ตารางอันดับ สามารถระบุชื่อที่ใช้และคะแนนสูงสุดไว้ในส่วนนี้ ทั้งนี้ไม่ใช่ข้อกำหนดของชิ้นงาน